In [ ]:
from google.colab import drive

# If something was mounted before, unmount it cleanly
try:
    drive.flush_and_unmount()
except Exception as e:
    print("flush_and_unmount warning:", e)

# Recreate the mount folder as EMPTY
!rm -rf /content/drive
!mkdir -p /content/drive

# Now mount
drive.mount("/content/drive")

In [ ]:
import os
import random
import torch
import librosa
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
%%writefile config.py
"""Centralized training/evaluation configuration.

All runtime code should *read* configuration values through the frozen
``Config`` object returned by ``get_config``. To change a value, edit this
file or supply a temporary override via CLI args and ``get_config``.
Runtime mutation of config attributes is blocked to avoid accidental drift.
"""

from dataclasses import dataclass, replace
import sys
import types
from typing import Any, Dict, Optional
import torch

# Paths (Colab/Drive)
BASE_PATH = "/content/drive/MyDrive/AudioDenoisingDrive"

# Root directory for processed spectrogram data
PROCESSED_ROOT = f"{BASE_PATH}/data/processed"
PROCESSED_DIR = PROCESSED_ROOT  # Alias for compatibility
PROCESSED_AUDIO_DIR = f"{BASE_PATH}/data/processed_audio"

# Directory to store checkpoints (Colab/Drive-friendly)
CHECKPOINT_DIR = f"{BASE_PATH}/checkpoints"

# Directory to store evaluation results (Colab/Drive-friendly)
RESULTS_DIR = f"{BASE_PATH}/results"

#Save visualized 

TRAIN_NOISY_DIR = f"{PROCESSED_ROOT}/train/noisy"
TRAIN_CLEAN_DIR = f"{PROCESSED_ROOT}/train/clean"

VAL_NOISY_DIR = f"{PROCESSED_ROOT}/val/noisy"
VAL_CLEAN_DIR = f"{PROCESSED_ROOT}/val/clean"
# Raw Data Paths
CLEAN_EN_DIR = f"{BASE_PATH}/data/raw/clean/clean_en"
CLEAN_EN_MP3_DIR = f"{BASE_PATH}/data/raw/clean/clean_en_mp3"
CLEAN_NP_DIR = f"{BASE_PATH}/data/raw/clean/clean_np"
CLEAN_NP_WEBM_DIR = f"{BASE_PATH}/data/raw/clean/clean_np_webm"
NOISE_DIR = f"{BASE_PATH}/data/raw/noise"

# Audio parameters

SAMPLE_RATE = 16000

N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024

FREQ_BINS = (N_FFT // 2) + 1   # 513 (raw STFT)
FREQ_BINS_MODEL = FREQ_BINS    # Model uses full 513-bin spectra

WINDOW_TYPE = "hann"
WINDOW = torch.hann_window(WIN_LENGTH)

# Preprocessing Config

NOISY_PAIRS_PER_CLEAN = 3             # Number of (noisy A, noisy B) pairs per clean clip
SNR_LIST = [-5, 0, 2.5, 5, 7.5, 10, 15]  # SNRs (dB) to sample from


# Spectrogram parameters

FIXED_TIME_FRAMES = 256        # must match preprocessing
USE_LOG_MAG = True


# Training parameters

BATCH_SIZE = 16  # Reduced for RTX 4050 6GB (UNet is memory-intensive)
ACCUMULATION_STEPS = 2  # Effective batch size = 4 * 8 = 32
GRADIENT_ACCUMULATION_STEPS = ACCUMULATION_STEPS
NUM_WORKERS = 2  # Set to 0 to avoid hanging in Colab/Jupyter parallel processing
PIN_MEMORY = True  # Pinning can add overhead on Colab's virtualized path
NOISE2NOISE = True  # When True, training targets are second noisy variants (Noise2Noise)
NUM_EPOCHS = 5

# Model architecture parameters
UNET_IN_CH = 1
UNET_OUT_CH = 1
UNET_BASE_CH = 64

# Optimizer and Scheduler parameters
INITIAL_LR = 2e-4
ADAM_BETAS = (0.9, 0.99)
USE_WEIGHT_DECAY = True
WEIGHT_DECAY = 1e-4
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
USE_SCHEDULER = True  # Flag to enable/disable scheduler


# Multi-Resolution STFT Loss parameters
LOSS_FUNCTION = "mse"  # Options: "mse", "l1", "mrstft", "hybrid", "hybrid_l1", "combined", "crm"
HYBRID_ALPHA = 0.5  # 0.5 means equal weight. < 0.5 favors MSE, > 0.5 favors MR-STFT
COMBINED_LAMBDA1 = 100.0  # Weight for waveform MSE Loss
COMBINED_LAMBDA2 = 1.0    # Weight for Multi-Resolution STFT Loss
CRM_MASK_ACTIVATION = "tanh"  # Activation for CRM mask components (recommended: tanh)

MR_STFT_FFT_SIZES = [1024, 2048, 512]
MR_STFT_HOP_SIZES = [120, 240, 50]
MR_STFT_WIN_LENGTHS = [600, 1200, 240]
MR_STFT_WINDOW = "hann_window"

# Griffin-Lim phase reconstruction parameters
GRIFFIN_LIM_ITER = 68          # Number of Griffin-Lim iterations

# Reconstruction policy and anti-hiss postfilter
PREFER_INPUT_PHASE_RECON = True   # If phase is available, prefer noisy-phase ISTFT over Griffin-Lim
APPLY_POSTFILTER = True           # Apply light postfilter to reduce static/hiss after reconstruction
POSTFILTER_CUTOFF_HZ = 6500.0     # Low-pass cutoff (Hz) for anti-hiss postfilter (was 7000)
POSTFILTER_STRENGTH = 0.35        # 0.0=no effect, 1.0=fully filtered (was 0.2)

# Spectral gating post-processor (attacks "rain" artifacts directly)
APPLY_SPECTRAL_GATE = True        # Zero-out quiet T-F bins below noise floor
SPECTRAL_GATE_THRESHOLD = 1.5     # Multiplier on estimated noise floor (higher = more aggressive)

# Wiener post-filter (soft SNR-based suppression)
APPLY_WIENER_POSTFILTER = True    # Apply Wiener-style post-filter after spectral gate
WIENER_BETA = 0.02                # Noise power estimate for Wiener; higher = more suppression

# Mask-based prediction mode (recommended for Stage 4+ training)
MASK_MODE = False                 # When True, UNet predicts a [0,1] ratio mask instead of magnitude


# Augmentation parameters

MIN_GAIN_DB = -10
MAX_GAIN_DB = 10
FREQ_MASK_PARAM = 20
TIME_MASK_PARAM = 30
NUM_FREQ_MASKS = 1
NUM_TIME_MASKS = 1


# Tools
FFMPEG_PATH = "ffmpeg"  # Assumes ffmpeg is in PATH (common for Linux)


# Visualization Defaults
VISUALIZATION_CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/unet_best.pt"


# -----------------------------------------------------------------------------
# Frozen config object and safe accessors
# -----------------------------------------------------------------------------

@dataclass(frozen=True)
class Config:
	SAMPLE_RATE: int
	N_FFT: int
	HOP_LENGTH: int
	WIN_LENGTH: int
	FREQ_BINS: int
	FREQ_BINS_MODEL: int
	WINDOW_TYPE: str

	NOISY_PAIRS_PER_CLEAN: int
	SNR_LIST: tuple

	FIXED_TIME_FRAMES: int
	USE_LOG_MAG: bool

	BATCH_SIZE: int
	ACCUMULATION_STEPS: int
	GRADIENT_ACCUMULATION_STEPS: int
	NUM_WORKERS: int
	PIN_MEMORY: bool
	NOISE2NOISE: bool
	NUM_EPOCHS: int

	UNET_IN_CH: int
	UNET_OUT_CH: int
	UNET_BASE_CH: int

	INITIAL_LR: float
	ADAM_BETAS: tuple
	USE_WEIGHT_DECAY: bool
	WEIGHT_DECAY: float
	SCHEDULER_PATIENCE: int
	SCHEDULER_FACTOR: float
	USE_SCHEDULER: bool

	LOSS_FUNCTION: str
	HYBRID_ALPHA: float
	COMBINED_LAMBDA1: float
	COMBINED_LAMBDA2: float
	CRM_MASK_ACTIVATION: str
	MR_STFT_FFT_SIZES: tuple
	MR_STFT_HOP_SIZES: tuple
	MR_STFT_WIN_LENGTHS: tuple
	MR_STFT_WINDOW: str

	GRIFFIN_LIM_ITER: int
	PREFER_INPUT_PHASE_RECON: bool
	APPLY_POSTFILTER: bool
	POSTFILTER_CUTOFF_HZ: float
	POSTFILTER_STRENGTH: float

	APPLY_SPECTRAL_GATE: bool
	SPECTRAL_GATE_THRESHOLD: float
	APPLY_WIENER_POSTFILTER: bool
	WIENER_BETA: float
	MASK_MODE: bool

	PROCESSED_ROOT: str
	PROCESSED_DIR: str
	PROCESSED_AUDIO_DIR: str
	CHECKPOINT_DIR: str
	RESULTS_DIR: str
	TRAIN_NOISY_DIR: str
	TRAIN_CLEAN_DIR: str
	VAL_NOISY_DIR: str
	VAL_CLEAN_DIR: str

	MIN_GAIN_DB: float
	MAX_GAIN_DB: float
	FREQ_MASK_PARAM: int
	TIME_MASK_PARAM: int
	NUM_FREQ_MASKS: int
	NUM_TIME_MASKS: int

	CLEAN_EN_DIR: str
	CLEAN_EN_MP3_DIR: str
	CLEAN_NP_DIR: str
	CLEAN_NP_WEBM_DIR: str
	NOISE_DIR: str

	FFMPEG_PATH: str

	VISUALIZATION_CHECKPOINT_PATH: str


CONFIG = Config(
	SAMPLE_RATE=SAMPLE_RATE,
	N_FFT=N_FFT,
	HOP_LENGTH=HOP_LENGTH,
	WIN_LENGTH=WIN_LENGTH,
	FREQ_BINS=FREQ_BINS,
	FREQ_BINS_MODEL=FREQ_BINS_MODEL,
	WINDOW_TYPE=WINDOW_TYPE,
	NOISY_PAIRS_PER_CLEAN=NOISY_PAIRS_PER_CLEAN,
	SNR_LIST=tuple(SNR_LIST),
	FIXED_TIME_FRAMES=FIXED_TIME_FRAMES,
	USE_LOG_MAG=USE_LOG_MAG,
	BATCH_SIZE=BATCH_SIZE,
	ACCUMULATION_STEPS=ACCUMULATION_STEPS,
	GRADIENT_ACCUMULATION_STEPS=GRADIENT_ACCUMULATION_STEPS,
	NUM_WORKERS=NUM_WORKERS,
	PIN_MEMORY=PIN_MEMORY,
	NOISE2NOISE=NOISE2NOISE,
	NUM_EPOCHS=NUM_EPOCHS,
	UNET_IN_CH=UNET_IN_CH,
	UNET_OUT_CH=UNET_OUT_CH,
	UNET_BASE_CH=UNET_BASE_CH,
	INITIAL_LR=INITIAL_LR,
	ADAM_BETAS=tuple(ADAM_BETAS),
	USE_WEIGHT_DECAY=USE_WEIGHT_DECAY,
	WEIGHT_DECAY=WEIGHT_DECAY,
	SCHEDULER_PATIENCE=SCHEDULER_PATIENCE,
	SCHEDULER_FACTOR=SCHEDULER_FACTOR,
	USE_SCHEDULER=USE_SCHEDULER,
	LOSS_FUNCTION=LOSS_FUNCTION,
	HYBRID_ALPHA=HYBRID_ALPHA,
	COMBINED_LAMBDA1=COMBINED_LAMBDA1,
	COMBINED_LAMBDA2=COMBINED_LAMBDA2,
	CRM_MASK_ACTIVATION=CRM_MASK_ACTIVATION,
	MR_STFT_FFT_SIZES=tuple(MR_STFT_FFT_SIZES),
	MR_STFT_HOP_SIZES=tuple(MR_STFT_HOP_SIZES),
	MR_STFT_WIN_LENGTHS=tuple(MR_STFT_WIN_LENGTHS),
	MR_STFT_WINDOW=MR_STFT_WINDOW,
	GRIFFIN_LIM_ITER=GRIFFIN_LIM_ITER,
	PREFER_INPUT_PHASE_RECON=PREFER_INPUT_PHASE_RECON,
	APPLY_POSTFILTER=APPLY_POSTFILTER,
	POSTFILTER_CUTOFF_HZ=POSTFILTER_CUTOFF_HZ,
	POSTFILTER_STRENGTH=POSTFILTER_STRENGTH,
	APPLY_SPECTRAL_GATE=APPLY_SPECTRAL_GATE,
	SPECTRAL_GATE_THRESHOLD=SPECTRAL_GATE_THRESHOLD,
	APPLY_WIENER_POSTFILTER=APPLY_WIENER_POSTFILTER,
	WIENER_BETA=WIENER_BETA,
	MASK_MODE=MASK_MODE,
	PROCESSED_ROOT=PROCESSED_ROOT,
	PROCESSED_DIR=PROCESSED_DIR,
	PROCESSED_AUDIO_DIR=PROCESSED_AUDIO_DIR,
	CHECKPOINT_DIR=CHECKPOINT_DIR,
	RESULTS_DIR=RESULTS_DIR,
	TRAIN_NOISY_DIR=TRAIN_NOISY_DIR,
	TRAIN_CLEAN_DIR=TRAIN_CLEAN_DIR,
	VAL_NOISY_DIR=VAL_NOISY_DIR,
	VAL_CLEAN_DIR=VAL_CLEAN_DIR,
	MIN_GAIN_DB=MIN_GAIN_DB,
	MAX_GAIN_DB=MAX_GAIN_DB,
	FREQ_MASK_PARAM=FREQ_MASK_PARAM,
	TIME_MASK_PARAM=TIME_MASK_PARAM,
	NUM_FREQ_MASKS=NUM_FREQ_MASKS,
	NUM_TIME_MASKS=NUM_TIME_MASKS,
	CLEAN_EN_DIR=CLEAN_EN_DIR,
	CLEAN_EN_MP3_DIR=CLEAN_EN_MP3_DIR,
	CLEAN_NP_DIR=CLEAN_NP_DIR,
	CLEAN_NP_WEBM_DIR=CLEAN_NP_WEBM_DIR,
	NOISE_DIR=NOISE_DIR,
	FFMPEG_PATH=FFMPEG_PATH,
	VISUALIZATION_CHECKPOINT_PATH=VISUALIZATION_CHECKPOINT_PATH,
)



def get_config(overrides: Optional[Dict[str, Any]] = None) -> Config:
	"""Return the frozen configuration object, optionally with overrides."""
	if overrides:
		return replace(CONFIG, **overrides)
	return CONFIG


In [ ]:
from config import (
    SAMPLE_RATE,
    N_FFT,
    HOP_LENGTH,
    WIN_LENGTH,
    WINDOW_TYPE,
    FIXED_TIME_FRAMES,
    USE_LOG_MAG,
    PROCESSED_ROOT,
    PROCESSED_DIR,
    PROCESSED_AUDIO_DIR,
    CLEAN_EN_DIR,
    CLEAN_NP_DIR,
    CLEAN_NP_WEBM_DIR,
    CLEAN_EN_MP3_DIR,
    NOISE_DIR,
    NOISY_PAIRS_PER_CLEAN,
    SNR_LIST,
    FFMPEG_PATH,
)

# ================= CONFIG =================
WINDOW = torch.hann_window(WIN_LENGTH)

# Create processed directories
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(PROCESSED_DIR, split, "clean"), exist_ok=True)
    os.makedirs(os.path.join(PROCESSED_DIR, split, "noisy"), exist_ok=True)


In [ ]:
def collect_files(directory, extensions=(".wav", ".mp3")):
    """Recursively collect files from directory and subfolders."""
    files = []
    for root, _, filenames in os.walk(directory):
        for f in filenames:
            if f.lower().endswith(extensions):
                files.append(os.path.join(root, f))
    return sorted(files)


def load_audio(path):
    """Load audio as 1D numpy array at SAMPLE_RATE."""
    audio, _ = librosa.load(path, sr=SAMPLE_RATE)
    return audio


def fix_length(audio, target_len=SAMPLE_RATE * 4):
    """Pad or trim audio to fixed length (default 4s).
    
    If shorter than target_len, the audio is looped (repeated) to fill the duration.
    This prevents 'silence padding' which wastes training capacity.
    """
    if len(audio) < target_len:
        # Loop the audio until it exceeds target_len
        n_repeats = int(np.ceil(target_len / len(audio)))
        audio = np.tile(audio, n_repeats)
    
    # Trim to exact target length
    audio = audio[:target_len]
    return audio


def add_noise(clean_audio, noise_audio, snr_db=5):
    """Mix noise with clean audio at given SNR."""
    noise_audio = noise_audio[:len(clean_audio)]
    clean_power = np.mean(clean_audio ** 2)
    noise_power = np.mean(noise_audio ** 2)
    target_noise_power = clean_power / (10 ** (snr_db / 10))
    noise_audio = noise_audio * np.sqrt(target_noise_power / (noise_power + 1e-8))
    return clean_audio + noise_audio

def compute_stft(audio):
    """Compute STFT magnitude and phase tensors (513 bins).

    Returns (mag, phase) where phase is angle radians; no log applied here.
    """
    waveform = torch.tensor(audio, dtype=torch.float32)
    stft = torch.stft(
        waveform,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=WINDOW,
        return_complex=True,
    )
    mag = torch.abs(stft)
    phase = torch.angle(stft)
    return mag, phase  # [513, T]


In [ ]:
# Collect files
clean_en_files = collect_files(CLEAN_EN_DIR, (".wav",))
clean_np_files = collect_files(CLEAN_NP_DIR, (".wav",))

# Randomly select 1200 audios
random.seed(42)  # for reproducibility

num_en_to_use = int(len(clean_en_files) * 0.75)
num_np_to_use = int(len(clean_np_files) * 0.60)

selected_en = random.sample(clean_en_files, num_en_to_use)
selected_np = random.sample(clean_np_files, num_np_to_use)

# Combine them with language labels
en_labeled = [(f, "en") for f in selected_en]
np_labeled = [(f, "np") for f in selected_np]
clean_files = en_labeled + np_labeled

print(f"English files selected: {len(selected_en)} (from {len(clean_en_files)})")
print(f"Nepali files selected: {len(selected_np)} (from {len(clean_np_files)})")
print(f"Total clean files selected: {len(clean_files)}")

# Noise 
noise_files = collect_files(NOISE_DIR, (".wav",))
if len(noise_files) == 0:
    raise ValueError("No noise files found in NOISE_DIR and subfolders!")


# Split 6:2:2 (adjusting test_size based on actual total count)
# To handle variable total counts, we use ratios for splitting instead of fixed numbers if the total size changes significantly.
# Original logic was fixed numbers (720 train, 240 val, 240 test = 1200 total).
# If correct total is different, we should probably stick to proportional splits.

# Let's use proportional split: 60% Train, 20% Val, 20% Test
train_files, test_val_files = train_test_split(clean_files, test_size=0.4, random_state=42)
val_files, test_files = train_test_split(test_val_files, test_size=0.5, random_state=42)

print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

Total clean files available: 2315
Total clean files selected: 1200
Train: 720 | Val: 240 | Test: 240


In [ ]:
def preprocess_and_save(clean_list, save_clean_dir, save_noisy_dir):
    idx_counter = 0
    for clean_path, lang in clean_list:
        clean_audio = load_audio(clean_path)
        clean_audio = fix_length(clean_audio)
        clean_mag, clean_phase = compute_stft(clean_audio)

        for pair_idx in range(NOISY_PAIRS_PER_CLEAN):
            # Noisy A
            noise_path_a = random.choice(noise_files)
            noise_audio_a = load_audio(noise_path_a)
            noise_audio_a = fix_length(noise_audio_a)
            snr_a = random.choice(SNR_LIST)
            noisy_audio_a = add_noise(clean_audio, noise_audio_a, snr_db=snr_a)
            noisy_mag_a, noisy_phase_a = compute_stft(noisy_audio_a)

            # Noisy B (different noise clip to encourage independence)
            noise_path_b = random.choice(noise_files)
            while noise_path_b == noise_path_a and len(noise_files) > 1:
                noise_path_b = random.choice(noise_files)
            noise_audio_b = load_audio(noise_path_b)
            noise_audio_b = fix_length(noise_audio_b)
            snr_b = random.choice(SNR_LIST)
            noisy_audio_b = add_noise(clean_audio, noise_audio_b, snr_db=snr_b)
            noisy_mag_b, noisy_phase_b = compute_stft(noisy_audio_b)

            clean_save_path = os.path.join(save_clean_dir, f"{idx_counter:05d}.pt")
            noisy_save_path = os.path.join(save_noisy_dir, f"{idx_counter:05d}.pt")

            # Save clean for evaluation/visualization; noisy contains paired variants for Noise2Noise
            torch.save((clean_mag, clean_phase), clean_save_path)
            torch.save(
                {
                    "a_mag": noisy_mag_a,
                    "a_phase": noisy_phase_a,
                    "b_mag": noisy_mag_b,
                    "b_phase": noisy_phase_b,
                },
                noisy_save_path,
            )

            idx_counter += 1

        if idx_counter % 10 == 0:
            print(f"Processed {idx_counter} pairs...")

    return idx_counter

train_count = preprocess_and_save(train_files, os.path.join(PROCESSED_DIR,"train/clean"), os.path.join(PROCESSED_DIR,"train/noisy"))
print(f"Saved {train_count} training pairs.")

val_count = preprocess_and_save(val_files, os.path.join(PROCESSED_DIR,"val/clean"), os.path.join(PROCESSED_DIR,"val/noisy"))
print(f"Saved {val_count} validation pairs.")

test_count = preprocess_and_save(test_files, os.path.join(PROCESSED_DIR,"test/clean"), os.path.join(PROCESSED_DIR,"test/noisy"))
print(f"Saved {test_count} test pairs.")

print("✅ Preprocessing complete. All splits saved with Noise2Noise pairs.")

In [ ]:
# Save noisy audio as .wav in dataset/processed_audio (Colab/Drive paths from config)
# Prepare directories for wav exports
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(PROCESSED_AUDIO_DIR, split, "clean"), exist_ok=True)
    os.makedirs(os.path.join(PROCESSED_AUDIO_DIR, split, "noisy"), exist_ok=True)


def preprocess_and_save_with_wav(clean_list, split):
    """Preprocess, save STFT tensors (Double-Noisy for N2N), and export wavs."""
    save_clean_dir = os.path.join(PROCESSED_DIR, split, "clean")
    save_noisy_dir = os.path.join(PROCESSED_DIR, split, "noisy")
    clean_wav_dir = os.path.join(PROCESSED_AUDIO_DIR, split, "clean")
    noisy_wav_dir = os.path.join(PROCESSED_AUDIO_DIR, split, "noisy")
    
    # Ensure all directories exist
    os.makedirs(save_clean_dir, exist_ok=True)
    os.makedirs(save_noisy_dir, exist_ok=True)
    os.makedirs(clean_wav_dir, exist_ok=True)
    os.makedirs(noisy_wav_dir, exist_ok=True)

    idx_counter = 0
    for clean_path, lang in clean_list:
        clean_audio = load_audio(clean_path)
        clean_audio = fix_length(clean_audio)
        clean_mag, clean_phase = compute_stft(clean_audio)

        # Iterate NOISY_PAIRS_PER_CLEAN times to generate multiple samples from one clean file
        for pair_idx in range(NOISY_PAIRS_PER_CLEAN):
            # Generate TWO independent noise realizations per sample for Noise2Noise
            
            # Realization A (Input)
            noise_path_a = random.choice(noise_files)
            noise_audio_a = load_audio(noise_path_a)
            noise_audio_a = fix_length(noise_audio_a)
            snr_a = random.choice(SNR_LIST)
            noisy_audio_a = add_noise(clean_audio, noise_audio_a, snr_db=snr_a)
            noisy_mag_a, noisy_phase_a = compute_stft(noisy_audio_a)
            
            # Realization B (Target)
            noise_path_b = random.choice(noise_files)
            noise_audio_b = load_audio(noise_path_b)
            noise_audio_b = fix_length(noise_audio_b)
            snr_b = random.choice(SNR_LIST)
            noisy_audio_b = add_noise(clean_audio, noise_audio_b, snr_db=snr_b)
            noisy_mag_b, noisy_phase_b = compute_stft(noisy_audio_b)

            # Save Tensors: Clean, plus BOTH noisy versions
            clean_save_path = os.path.join(save_clean_dir, f"{idx_counter:05d}.pt")
            noisy_save_path = os.path.join(save_noisy_dir, f"{idx_counter:05d}.pt")
            
            # Save clean as usual
            torch.save((clean_mag, clean_phase), clean_save_path)
            # Save tuple of 4 for N2N (or just noisy A if standard, but we save 4 to be flexible)
            torch.save((noisy_mag_a, noisy_phase_a, noisy_mag_b, noisy_phase_b), noisy_save_path)

            # Export WAVs (A is strictly input, B is target)
            snr_tag_a = str(snr_a).replace(".", "p").replace("-", "m")
            clean_wav_path = os.path.join(clean_wav_dir, f"{idx_counter:05d}.wav")
            noisy_wav_path_a = os.path.join(noisy_wav_dir, f"{idx_counter:05d}_A_snr{snr_tag_a}.wav")
            
            sf.write(clean_wav_path, clean_audio, SAMPLE_RATE)
            sf.write(noisy_wav_path_a, noisy_audio_a, SAMPLE_RATE)
            # We don't necessarily save wav B unless debugging, to save space

            idx_counter += 1

        if idx_counter % 10 == 0:
            print(f"Processed {idx_counter} pairs for {split}...")

    return idx_counter


# Run this instead of preprocess_and_save if you want .wav outputs
train_wav_count = preprocess_and_save_with_wav(train_files, "train")
val_wav_count = preprocess_and_save_with_wav(val_files, "val")
test_wav_count = preprocess_and_save_with_wav(test_files, "test")
print(f"Saved WAVs: train={train_wav_count}, val={val_wav_count}, test={test_wav_count}")


In [ ]:
from config import SAMPLE_RATE

def calculate_snr(clean_signal, noisy_signal):
    """
    Calculate the Signal-to-Noise Ratio (SNR) in dB.
    SNR = 10 * log10(P_signal / P_noise)
    """
    signal_power = np.mean(clean_signal ** 2)
    noise_power = np.mean((clean_signal - noisy_signal) ** 2)
    if noise_power == 0:
        return float('inf')  # Infinite SNR if no noise
    return 10 * np.log10(signal_power / noise_power)

def analyze_snr_distribution(clean_dir, noisy_dir):
    """
    Analyze the SNR distribution for a dataset.
    Args:
        clean_dir (str): Path to the directory containing clean audio files.
        noisy_dir (str): Path to the directory containing noisy audio files.
    """
    snr_values = []

    # Ensure both directories exist
    if not os.path.exists(clean_dir) or not os.path.exists(noisy_dir):
        print("Error: One or both directories do not exist.")
        return

    # Iterate through clean audio files
    for file_name in os.listdir(clean_dir):
        clean_path = os.path.join(clean_dir, file_name)
        noisy_path = os.path.join(noisy_dir, file_name)

        # Check if corresponding noisy file exists
        if not os.path.exists(noisy_path):
            print(f"Warning: No matching noisy file for {file_name}. Skipping...")
            continue

        # Load clean and noisy audio
        clean_signal, sr_clean = sf.read(clean_path)
        noisy_signal, sr_noisy = sf.read(noisy_path)

        # Ensure sampling rates match
        if sr_clean != SAMPLE_RATE or sr_noisy != SAMPLE_RATE:
            print(f"Error: Sampling rates do not match for {file_name}. Skipping...")
            continue

        # Calculate SNR
        snr = calculate_snr(clean_signal, noisy_signal)
        snr_values.append(snr)

    # Analyze SNR distribution
    if snr_values:
        print(f"Number of samples analyzed: {len(snr_values)}")
        print(f"Average SNR: {np.mean(snr_values):.2f} dB")
        print(f"Median SNR: {np.median(snr_values):.2f} dB")
        print(f"Min SNR: {np.min(snr_values):.2f} dB")
        print(f"Max SNR: {np.max(snr_values):.2f} dB")

        # Plot histogram
        plt.hist(snr_values, bins=20, color='blue', edgecolor='black', alpha=0.7)
        plt.title("SNR Distribution")
        plt.xlabel("SNR (dB)")
        plt.ylabel("Frequency")
        plt.grid(True)
        plt.show()
    else:
        print("No valid SNR values calculated. Please check your dataset.")

# Example usage
clean_audio_dir = os.path.join(PROCESSED_AUDIO_DIR, "val/clean")  # Path to clean audio files
noisy_audio_dir = os.path.join(PROCESSED_AUDIO_DIR, "val/noisy")  # Path to noisy audio files

analyze_snr_distribution(clean_audio_dir, noisy_audio_dir)

In [ ]:
# Analyze SNR for test files
clean_test_audio_dir = os.path.join(PROCESSED_AUDIO_DIR, "test/clean")  # Path to clean test audio files
noisy_test_audio_dir = os.path.join(PROCESSED_AUDIO_DIR, "test/noisy")  # Path to noisy test audio files

analyze_snr_distribution(clean_test_audio_dir, noisy_test_audio_dir)

In [ ]:
%%writefile dataset.py
# dataset.py
import os
import random
from typing import Optional, Tuple

import librosa
import numpy as np
import torch
from torch.utils.data import Dataset

from config import get_config


# Helpers for dynamic mixing
def load_audio(path: str, sr: int) -> np.ndarray:
    """Load audio as 1D numpy array with the provided sample rate."""

    audio, _ = librosa.load(path, sr=sr)
    return audio


def fix_length(audio: np.ndarray, target_len: int) -> np.ndarray:
    """Pad or trim audio to a fixed length (loops if shorter)."""

    if len(audio) < target_len:
        n_repeats = int(np.ceil(target_len / len(audio)))
        audio = np.tile(audio, n_repeats)
    return audio[:target_len]


def add_noise(clean_audio: np.ndarray, noise_audio: np.ndarray, snr_db: float = 5) -> np.ndarray:
    """Mix noise with clean audio at the specified SNR."""

    noise_audio = noise_audio[:len(clean_audio)]
    clean_power = np.mean(clean_audio ** 2)
    noise_power = np.mean(noise_audio ** 2)
    target_noise_power = clean_power / (10 ** (snr_db / 10))
    noise_audio = noise_audio * np.sqrt(target_noise_power / (noise_power + 1e-8))
    return clean_audio + noise_audio


def compute_stft_tensor(
    audio: np.ndarray,
    *,
    n_fft: int,
    hop_length: int,
    win_length: int,
    window: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute STFT magnitude and phase tensors using provided parameters."""

    waveform = torch.tensor(audio, dtype=torch.float32)
    if window is None:
        window = torch.hann_window(win_length)
    stft = torch.stft(
        waveform,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        return_complex=True,
    )
    mag = torch.abs(stft)
    phase = torch.angle(stft)
    return mag, phase

class DenoisingDataset(Dataset):
    """
    Dataset for audio denoising using precomputed STFT tensors (.pt files).
    (Legacy support for static files)
    """
    def __init__(self, root_dir, split="train", return_phase: bool = False, target: str = "clean", cfg=None):
        """
        root_dir : str : path to processed folder
        split    : str : "train", "val", or "test"
        target   : "clean" for supervised denoising, "noisy" for Noise2Noise (input noisy A, target noisy B)
        cfg      : Config : optional immutable config to share overrides across components
        """
        assert split in ["train", "val", "test"], "split must be train/val/test"
        assert target in ["clean", "noisy"], "target must be 'clean' or 'noisy'"

        self.cfg = cfg or get_config()
        self.clean_dir = os.path.join(root_dir, split, "clean")
        self.noisy_dir = os.path.join(root_dir, split, "noisy")
        self.split = split
        self.return_phase = return_phase
        self.target = target

        # Cached config values for readability
        self.fixed_frames = self.cfg.FIXED_TIME_FRAMES
        self.use_log_mag = self.cfg.USE_LOG_MAG
        self.freq_bins = self.cfg.FREQ_BINS

        # List all files and sort to ensure clean/noisy pairing
        self.clean_files = sorted([f for f in os.listdir(self.clean_dir) if f.endswith(".pt")])
        self.noisy_files = sorted([f for f in os.listdir(self.noisy_dir) if f.endswith(".pt")])

        # Sanity check
        assert len(self.clean_files) == len(self.noisy_files), \
            f"Mismatch: {len(self.clean_files)} clean vs {len(self.noisy_files)} noisy files"

        self.length = len(self.clean_files)

    def __len__(self):
        return self.length

    def random_gain(self, mag):
        gain_db = torch.empty(1).uniform_(self.cfg.MIN_GAIN_DB, self.cfg.MAX_GAIN_DB).item()
        gain = 10 ** (gain_db / 20)
        return mag * gain

    def spec_augment(self, mag):
        augmented_mag = mag.clone()
        freq_bins, time_steps = augmented_mag.shape
        
        # Frequency Masking
        for _ in range(self.cfg.NUM_FREQ_MASKS):
            f = int(torch.empty(1).uniform_(0, self.cfg.FREQ_MASK_PARAM).item())
            if f > 0:
                f0 = int(torch.empty(1).uniform_(0, freq_bins - f).item())
                augmented_mag[f0:f0 + f, :] = 0.0
                
        # Time Masking
        for _ in range(self.cfg.NUM_TIME_MASKS):
            t = int(torch.empty(1).uniform_(0, self.cfg.TIME_MASK_PARAM).item())
            if t > 0:
                t0 = int(torch.empty(1).uniform_(0, time_steps - t).item())
                augmented_mag[:, t0:t0 + t] = 0.0
                
        return augmented_mag

    def __getitem__(self, idx):
        clean_path = os.path.join(self.clean_dir, self.clean_files[idx])
        noisy_path = os.path.join(self.noisy_dir, self.noisy_files[idx])

        loaded_clean = torch.load(clean_path, weights_only=False)
        loaded_noisy = torch.load(noisy_path, weights_only=False)

        # Support legacy single-tensor saves (mag only) and new (mag, phase) tuples
        if isinstance(loaded_clean, torch.Tensor):
            clean = loaded_clean.float()
            clean_phase = None
        else:
            # expect (mag, phase) or {'mag':..., 'phase':...}
            if isinstance(loaded_clean, dict):
                clean = loaded_clean['mag'].float()
                clean_phase = loaded_clean.get('phase', None)
            else:
                clean, clean_phase = loaded_clean
                clean = clean.float()

        noisy_phase_a = None
        noisy_phase_b = None

        # Allow saved formats:
        # - Tensor (single noisy) -> duplicated for Noise2Noise target
        # - Tuple (mag, phase)
        # - Tuple (noisyA_mag, noisyA_phase, noisyB_mag, noisyB_phase)
        # - Dict with keys a_mag, b_mag, a_phase, b_phase
        if isinstance(loaded_noisy, torch.Tensor):
            noisy_a = loaded_noisy.float()
            noisy_b = loaded_noisy.float()
        elif isinstance(loaded_noisy, dict):
            if "a_mag" in loaded_noisy and "b_mag" in loaded_noisy:
                noisy_a = loaded_noisy["a_mag"].float()
                noisy_b = loaded_noisy["b_mag"].float()
                noisy_phase_a = loaded_noisy.get("a_phase", None)
                noisy_phase_b = loaded_noisy.get("b_phase", None)
            else:
                noisy = loaded_noisy['mag'].float()
                noisy_phase_a = loaded_noisy.get('phase', None)
                noisy_a = noisy
                noisy_b = noisy
                noisy_phase_b = noisy_phase_a
        else:
            if len(loaded_noisy) == 2:
                noisy, noisy_phase = loaded_noisy
                noisy = noisy.float()
                noisy_a = noisy
                noisy_b = noisy
                noisy_phase_a = noisy_phase
                noisy_phase_b = noisy_phase
            elif len(loaded_noisy) == 4:
                noisy_a, noisy_phase_a, noisy_b, noisy_phase_b = loaded_noisy
                noisy_a = noisy_a.float()
                noisy_b = noisy_b.float()
            else:
                raise ValueError("Unsupported noisy tensor format")

        # Safety checks
        assert clean.shape[0] == self.freq_bins, f"Clean tensor has {clean.shape[0]} freq bins, expected {self.freq_bins}"
        assert noisy_a.shape[0] == self.freq_bins, f"Noisy tensor A has {noisy_a.shape[0]} freq bins, expected {self.freq_bins}"
        assert noisy_b.shape[0] == self.freq_bins, f"Noisy tensor B has {noisy_b.shape[0]} freq bins, expected {self.freq_bins}"

        # Pad or crop time dimension to fixed size
        if self.fixed_frames is not None:
            def _pad_crop(spec, phase):
                if spec.shape[1] < self.fixed_frames:
                    spec = torch.nn.functional.pad(spec, (0, self.fixed_frames - spec.shape[1]))
                    if phase is not None:
                        phase = torch.nn.functional.pad(phase, (0, self.fixed_frames - phase.shape[1]))
                else:
                    spec = spec[:, :self.fixed_frames]
                    if phase is not None:
                        phase = phase[:, :self.fixed_frames]
                return spec, phase

            clean, clean_phase = _pad_crop(clean, clean_phase)
            noisy_a, noisy_phase_a = _pad_crop(noisy_a, noisy_phase_a)
            noisy_b, noisy_phase_b = _pad_crop(noisy_b, noisy_phase_b)

        # Apply Augmentation (Only strictly if split is training)
        if self.split == 'train':
            noisy_a = self.random_gain(noisy_a)
            noisy_a = self.spec_augment(noisy_a)

        # Optional log-magnitude (note: phases are stored as angles; leave unchanged)
        if self.use_log_mag:
            clean = torch.log1p(clean)
            noisy_a = torch.log1p(noisy_a)
            noisy_b = torch.log1p(noisy_b)

        # Add channel dimension for CNN
        clean = clean.unsqueeze(0)  # (1, freq_bins, T)
        noisy_a = noisy_a.unsqueeze(0)
        noisy_b = noisy_b.unsqueeze(0)

        # Select target for training/eval
        target_mag = clean if self.target == "clean" else noisy_b

        if self.return_phase:
            # return phase without channel dim so batch shape is (B, freq, T)
            def _phase_or_dummy(phase):
                if phase is None:
                    return torch.zeros(clean.shape[1], clean.shape[2])
                return phase

            noisy_phase_a = _phase_or_dummy(noisy_phase_a)
            noisy_phase_b = _phase_or_dummy(noisy_phase_b)
            clean_phase = _phase_or_dummy(clean_phase)

            return noisy_a, target_mag, noisy_phase_a, noisy_phase_b if self.target == "noisy" else clean_phase

        return noisy_a, target_mag
class DynamicDenoisingDataset(Dataset):
    """
    Dataset for on-the-fly audio denoising.
    Mixes clean speech and noise dynamically with random SNR.
    """
    def __init__(self, clean_files, noise_files, split="train", return_phase=False, target="clean", cfg=None):
        self.cfg = cfg or get_config()
        self.clean_files = clean_files
        self.noise_files = noise_files
        self.split = split
        self.return_phase = return_phase
        self.target = target
        self.window = torch.hann_window(self.cfg.WIN_LENGTH)
        self.target_len = self.cfg.SAMPLE_RATE * 4

    def __len__(self):
        return len(self.clean_files)

    def random_gain(self, mag):
        gain_db = torch.empty(1).uniform_(self.cfg.MIN_GAIN_DB, self.cfg.MAX_GAIN_DB).item()
        gain = 10 ** (gain_db / 20)
        return mag * gain

    def spec_augment(self, mag):
        augmented_mag = mag.clone()
        freq_bins, time_steps = augmented_mag.shape
        for _ in range(self.cfg.NUM_FREQ_MASKS):
            f = int(torch.empty(1).uniform_(0, self.cfg.FREQ_MASK_PARAM).item())
            if f > 0:
                f0 = int(torch.empty(1).uniform_(0, freq_bins - f).item())
                augmented_mag[f0:f0 + f, :] = 0.0
        for _ in range(self.cfg.NUM_TIME_MASKS):
            t = int(torch.empty(1).uniform_(0, self.cfg.TIME_MASK_PARAM).item())
            if t > 0:
                t0 = int(torch.empty(1).uniform_(0, time_steps - t).item())
                augmented_mag[:, t0:t0 + t] = 0.0
        return augmented_mag

    def __getitem__(self, idx):
        clean_path = self.clean_files[idx]
        
        # Load Clean
        clean_audio = load_audio(clean_path, sr=self.cfg.SAMPLE_RATE)
        clean_audio = fix_length(clean_audio, target_len=self.target_len)

        # -------------------------------------------------------
        # Path 1: Noise2Noise (Target = Second Noisy Version)
        # -------------------------------------------------------
        if self.target == "noisy":
            # Generate Noisy A (Input)
            noise_path_a = random.choice(self.noise_files)
            noise_audio_a = fix_length(load_audio(noise_path_a, sr=self.cfg.SAMPLE_RATE), target_len=self.target_len)
            snr_a = random.choice(self.cfg.SNR_LIST) if self.split == "train" else 5.0
            noisy_audio_a = add_noise(clean_audio, noise_audio_a, snr_a)

            # Generate Noisy B (Target) - MUST be a different noise file for independence
            # Correlated noise between A and B causes residual static/rain artifacts
            remaining = [p for p in self.noise_files if p != noise_path_a]
            noise_path_b = random.choice(remaining) if remaining else random.choice(self.noise_files)
            noise_audio_b = fix_length(load_audio(noise_path_b, sr=self.cfg.SAMPLE_RATE), target_len=self.target_len)
            snr_b = random.choice(self.cfg.SNR_LIST) if self.split == "train" else 5.0
            noisy_audio_b = add_noise(clean_audio, noise_audio_b, snr_b)

            # Compute STFTs
            noisy_mag_a, noisy_phase_a = compute_stft_tensor(
                noisy_audio_a,
                n_fft=self.cfg.N_FFT,
                hop_length=self.cfg.HOP_LENGTH,
                win_length=self.cfg.WIN_LENGTH,
                window=self.window,
            )
            noisy_mag_b, noisy_phase_b = compute_stft_tensor(
                noisy_audio_b,
                n_fft=self.cfg.N_FFT,
                hop_length=self.cfg.HOP_LENGTH,
                win_length=self.cfg.WIN_LENGTH,
                window=self.window,
            )

            input_mag = noisy_mag_a
            target_mag = noisy_mag_b
            input_phase = noisy_phase_a
            target_phase = noisy_phase_b
        
        # -------------------------------------------------------
        # Path 2: Standard Supervised (Target = Clean)
        # -------------------------------------------------------
        else:
            noise_path = random.choice(self.noise_files)
            noise_audio = fix_length(load_audio(noise_path, sr=self.cfg.SAMPLE_RATE), target_len=self.target_len)
            snr = random.choice(self.cfg.SNR_LIST) if self.split == "train" else 5.0
            noisy_audio = add_noise(clean_audio, noise_audio, snr)

            input_mag, input_phase = compute_stft_tensor(
                noisy_audio,
                n_fft=self.cfg.N_FFT,
                hop_length=self.cfg.HOP_LENGTH,
                win_length=self.cfg.WIN_LENGTH,
                window=self.window,
            )
            clean_mag, clean_phase = compute_stft_tensor(
                clean_audio,
                n_fft=self.cfg.N_FFT,
                hop_length=self.cfg.HOP_LENGTH,
                win_length=self.cfg.WIN_LENGTH,
                window=self.window,
            )
            
            input_mag = input_mag
            target_mag = clean_mag
            target_phase = clean_phase

        # Common Post-Processing --------------------------------

        # Pad or crop time dimension
        if self.cfg.FIXED_TIME_FRAMES is not None:
            def _pad_crop_spec(spec, phase):
                if spec.shape[1] < self.cfg.FIXED_TIME_FRAMES:
                    spec = torch.nn.functional.pad(spec, (0, self.cfg.FIXED_TIME_FRAMES - spec.shape[1]))
                    if phase is not None:
                        phase = torch.nn.functional.pad(phase, (0, self.cfg.FIXED_TIME_FRAMES - phase.shape[1]))
                else:
                    spec = spec[:, :self.cfg.FIXED_TIME_FRAMES]
                    if phase is not None:
                        phase = phase[:, :self.cfg.FIXED_TIME_FRAMES]
                return spec, phase

            input_mag, input_phase = _pad_crop_spec(input_mag, input_phase)
            target_mag, target_phase = _pad_crop_spec(target_mag, target_phase)

        # Augment Input (Only Training)
        if self.split == "train":
            input_mag = self.random_gain(input_mag)
            input_mag = self.spec_augment(input_mag)

        # Log Transform
        if self.cfg.USE_LOG_MAG:
            input_mag = torch.log1p(input_mag)
            target_mag = torch.log1p(target_mag)

        # Add Channel Dim
        input_mag = input_mag.unsqueeze(0)
        target_mag = target_mag.unsqueeze(0)

        if self.return_phase:
            return input_mag, target_mag, input_phase, target_phase
        
        return input_mag, target_mag


In [ ]:
%%writefile model.py
# model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple
from config import N_FFT, HOP_LENGTH, WIN_LENGTH, WINDOW


def crop_to_match(encoder_feat: torch.Tensor, decoder_feat: torch.Tensor) -> torch.Tensor:
    """Center-crop or pad encoder feature to match decoder spatial size for skip connections."""
    diff_y = encoder_feat.size(2) - decoder_feat.size(2)
    diff_x = encoder_feat.size(3) - decoder_feat.size(3)
    # Crop if encoder is larger, pad if smaller
    if diff_y >= 0 and diff_x >= 0:
        return encoder_feat[:, :, diff_y // 2 : encoder_feat.size(2) - (diff_y - diff_y // 2), diff_x // 2 : encoder_feat.size(3) - (diff_x - diff_x // 2)]
    # Pad order: (left, right, top, bottom)
    pad = [max(-diff_x // 2, 0), max(-diff_x - (-diff_x // 2), 0), max(-diff_y // 2, 0), max(-diff_y - (-diff_y // 2), 0)]
    return F.pad(encoder_feat, pad)

# Double Convolution Block from the original U-Net (Ronneberger et al. 2015)
class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.insert(3, nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class Encoder(nn.Module):
    """Four-level encoder with max-pooling, matching spectrogram dimensions (513 x T)."""
    def __init__(self, in_ch: int = 1, base_ch: int = 64, dropout: float = 0.0):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base_ch, dropout=dropout)
        self.enc2 = DoubleConv(base_ch, base_ch * 2, dropout=dropout)
        self.enc3 = DoubleConv(base_ch * 2, base_ch * 4, dropout=dropout)
        self.enc4 = DoubleConv(base_ch * 4, base_ch * 8, dropout=dropout)
        self.center = DoubleConv(base_ch * 8, base_ch * 16, dropout=dropout)
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x: torch.Tensor):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        x4 = self.enc4(self.pool(x3))
        x5 = self.center(self.pool(x4))
        return x1, x2, x3, x4, x5

class Decoder(nn.Module):
    def __init__(self, out_ch: int = 1, base_ch: int = 64, dropout: float = 0.0, final_activation: str = "softplus"):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(base_ch * 16, base_ch * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base_ch * 16, base_ch * 8, dropout=dropout)

        self.up3 = nn.ConvTranspose2d(base_ch * 8, base_ch * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_ch * 8, base_ch * 4, dropout=dropout)

        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_ch * 4, base_ch * 2, dropout=dropout)

        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_ch * 2, base_ch, dropout=dropout)

        self.final = nn.Conv2d(base_ch, out_ch, kernel_size=1)
        self.final_activation = final_activation

    def forward(self, x1: torch.Tensor, x2: torch.Tensor, x3: torch.Tensor, x4: torch.Tensor, x5: torch.Tensor) -> torch.Tensor:
        d4 = self.up4(x5)
        d4 = self.dec4(torch.cat([d4, crop_to_match(x4, d4)], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, crop_to_match(x3, d3)], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, crop_to_match(x2, d2)], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, crop_to_match(x1, d1)], dim=1))

        out = self.final(d1)
        if self.final_activation == "softplus":
            return F.softplus(out)
        if self.final_activation == "relu":
            return F.relu(out)
        return out

class UNet(nn.Module):
    """Standard U-Net for magnitude spectrogram denoising (Ronneberger et al. 2015).

    When ``mask_mode=True`` the network predicts a [0, 1] ratio mask (via sigmoid)
    that is element-wise multiplied with the input spectrogram.  This guarantees
    the output can reach exactly zero (full suppression) and never exceed the
    input energy — eliminating the residual hiss that softplus cannot remove.
    """
    def __init__(self, in_ch: int = 1, out_ch: int = 1, base_ch: int = 64,
                 dropout: float = 0.0, final_activation: str = "softplus",
                 mask_mode: bool = False):
        super().__init__()
        self.mask_mode = mask_mode
        self.encoder = Encoder(in_ch=in_ch, base_ch=base_ch, dropout=dropout)
        # In mask mode, the final activation is handled manually (sigmoid)
        dec_activation = "none" if mask_mode else final_activation
        self.decoder = Decoder(out_ch=out_ch, base_ch=base_ch, dropout=dropout, final_activation=dec_activation)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pad input to be divisible by 16 (2^4) because of 4 pooling layers
        h, w = x.shape[2], x.shape[3]
        pad_h = (16 - (h % 16)) % 16
        pad_w = (16 - (w % 16)) % 16
        
        if pad_h > 0 or pad_w > 0:
            x_padded = F.pad(x, (0, pad_w, 0, pad_h))
        else:
            x_padded = x

        x1, x2, x3, x4, x5 = self.encoder(x_padded)
        out = self.decoder(x1, x2, x3, x4, x5)

        # Crop back to original size if padded
        if pad_h > 0 or pad_w > 0:
            out = out[..., :h, :w]

        if self.mask_mode:
            # Predict a ratio mask in [0, 1] and apply it to the input
            mask = torch.sigmoid(out)
            return mask * x
            
        return out


def stft_transform(waveform: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute magnitude and phase; waveform shape (T,)."""
    stft = torch.stft(
        waveform,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=WINDOW.to(waveform.device),
        return_complex=True,
    )
    mag = torch.abs(stft)
    phase = torch.angle(stft)
    return mag, phase

# Kaiming (He) Initialization for UNet weights
def init_kaiming(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)




Overwriting model.py


In [ ]:
%%writefile MR_STFT.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class SpectralConvergenceLoss(nn.Module):
    """Spectral Convergence Loss."""
    def __init__(self):
        super().__init__()

    def forward(self, x_mag, y_mag):
        """
        x_mag: Magnitude spectrogram of predicted waveform
        y_mag: Magnitude spectrogram of target waveform
        """
        return torch.norm(y_mag - x_mag, p="fro") / (torch.norm(y_mag, p="fro") + 1e-7)


class LogSTFTMagnitudeLoss(nn.Module):
    """Log STFT Magnitude Loss."""
    def __init__(self):
        super().__init__()

    def forward(self, x_mag, y_mag):
        """
        x_mag: Magnitude spectrogram of predicted waveform
        y_mag: Magnitude spectrogram of target waveform
        """
        return F.l1_loss(torch.log(x_mag + 1e-7), torch.log(y_mag + 1e-7))


class STFTLoss(nn.Module):
    """
    Computes the STFT loss for a single resolution.
    Consists of Spectral Convergence Loss (SC) and Log STFT Magnitude Loss (Mag).
    """
    def __init__(self, fft_size=1024, hop_size=120, win_length=600, window="hann_window"):
        super().__init__()
        self.fft_size = fft_size
        self.hop_size = hop_size
        self.win_length = win_length
        # Register window as buffer to be moved to GPU automatically
        self.register_buffer("window", getattr(torch, window)(win_length))
        self.sc_loss = SpectralConvergenceLoss()
        self.mag_loss = LogSTFTMagnitudeLoss()

    def forward(self, x, y):
        """
        x: Predicted waveform (B, T) or (B, 1, T)
        y: Target waveform (B, T) or (B, 1, T)
        """
        # Ensure (B, T)
        if x.dim() == 3: x = x.squeeze(1)
        if y.dim() == 3: y = y.squeeze(1)

        # Compute STFT
        # center=True is default and important for alignment
        x_stft = torch.stft(x, n_fft=self.fft_size, hop_length=self.hop_size, 
                            win_length=self.win_length, window=self.window, 
                            return_complex=True)
        y_stft = torch.stft(y, n_fft=self.fft_size, hop_length=self.hop_size, 
                            win_length=self.win_length, window=self.window, 
                            return_complex=True)

        # Magnitudes
        x_mag = torch.abs(x_stft)
        y_mag = torch.abs(y_stft)

        # Spectral Convergence Loss
        sc_l = self.sc_loss(x_mag, y_mag)

        # Log STFT Magnitude Loss
        mag_l = self.mag_loss(x_mag, y_mag)

        return sc_l, mag_l


class MultiResolutionSTFTLoss(nn.Module):
    """
    Compute STFT loss at multiple resolutions to capture both time and frequency structures.
    """
    def __init__(self,
                 fft_sizes=[512, 1024, 2048],
                 hop_sizes=[50, 120, 240],
                 win_lengths=[240, 600, 1200],
                 window="hann_window",
                 factor_sc=0.1,
                 factor_mag=0.1):
        super().__init__()
        assert len(fft_sizes) == len(hop_sizes) == len(win_lengths)

        self.stft_losses = nn.ModuleList()
        for fs, ss, wl in zip(fft_sizes, hop_sizes, win_lengths):
            self.stft_losses.append(STFTLoss(fs, ss, wl, window))
        
        self.factor_sc = factor_sc
        self.factor_mag = factor_mag

    def forward(self, x, y):
        sc_loss = 0.0
        mag_loss = 0.0
        for f in self.stft_losses:
            sc, mag = f(x, y)
            sc_loss += sc
            mag_loss += mag
        
        sc_loss /= len(self.stft_losses)
        mag_loss /= len(self.stft_losses)

        return self.factor_sc * sc_loss + self.factor_mag * mag_loss


class CombinedLoss(nn.Module):
    """
    Combines Waveform MSE Loss and Multi-Resolution STFT Loss.
    L_total = lambda1 * MSE + lambda2 * MRSTFT
    """
    def __init__(self, 
                 lambda1=100.0, 
                 lambda2=1.0,
                 fft_sizes=[512, 1024, 2048],
                 hop_sizes=[50, 120, 240],
                 win_lengths=[240, 600, 1200],
                 window="hann_window"):
        super().__init__()
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.mse_loss = nn.MSELoss()
        self.mrstft_loss = MultiResolutionSTFTLoss(fft_sizes, hop_sizes, win_lengths, window)

    def forward(self, pred_wav, target_wav):
        """
        pred_wav: Predicted waveform (B, 1, T) or (B, T)
        target_wav: Target waveform (B, 1, T) or (B, T)
        """
        mse = self.mse_loss(pred_wav, target_wav)
        mrstft = self.mrstft_loss(pred_wav, target_wav)
        
        return self.lambda1 * mse + self.lambda2 * mrstft, mse, mrstft


In [2]:
%%writefile train.py
# import warnings
# warnings.filterwarnings('ignore', category=FutureWarning, module='torch')

import os
import argparse
import logging
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from config import get_config
from dataset import DenoisingDataset
from utilis import reconstruct_waveform_from_mag_and_phase
from model import UNet, init_kaiming

try:
    from MR_STFT import MultiResolutionSTFTLoss, CombinedLoss
except ImportError:
    # Fallback if MR_STFT.py is missing or broken, though we expect it to be there
    MultiResolutionSTFTLoss = None
    CombinedLoss = None


cfg = get_config()


def setup_logger(log_dir: str) -> logging.Logger:
    """Configure a logger that writes to both console and a timestamped log file."""
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"train_{timestamp}.log")

    logger = logging.getLogger("trainer")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()  # avoid duplicate handlers on re-runs

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

    # File handler
    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.INFO)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    logger.info(f"Log file: {log_file}")
    return logger


class PlaceholderMRSTFTLoss(nn.Module):
    """Stand-in for Multi-Resolution STFT loss; uses simple L1 on magnitudes."""

    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return self.l1(pred, target)


class HybridLoss(nn.Module):
    """Combines MR-STFT and MSE loss."""
    def __init__(self, mrstft_loss, alpha=0.5):
        super().__init__()
        self.mrstft = mrstft_loss
        self.mse = nn.MSELoss()
        self.alpha = alpha

    def forward(self, pred_mag, target_mag, pred_wav, target_wav):
        loss_mrstft = self.mrstft(pred_wav, target_wav)
        loss_mse = self.mse(pred_mag, target_mag)
        # alpha controls MR-STFT contribution
        # If alpha=0.3, then: 0.3 * MRSTFT + 0.7 * MSE
        return self.alpha * loss_mrstft + (1 - self.alpha) * loss_mse


class HybridL1Loss(nn.Module):
    """Combines MR-STFT and L1 loss.
    
    L1 (median-seeking) is more robust to noisy targets than MSE (mean-seeking),
    making this better suited for Noise2Noise training where targets contain noise.
    """
    def __init__(self, mrstft_loss, alpha=0.5):
        super().__init__()
        self.mrstft = mrstft_loss
        self.l1 = nn.L1Loss()
        self.alpha = alpha

    def forward(self, pred_mag, target_mag, pred_wav, target_wav):
        loss_mrstft = self.mrstft(pred_wav, target_wav)
        loss_l1 = self.l1(pred_mag, target_mag)
        return self.alpha * loss_mrstft + (1 - self.alpha) * loss_l1


class CRMLoss(nn.Module):
    """Complex Ratio Mask loss on real/imag components."""

    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, pred_complex: torch.Tensor, target_complex: torch.Tensor) -> torch.Tensor:
        return self.mse(pred_complex.real, target_complex.real) + self.mse(pred_complex.imag, target_complex.imag)


def apply_crm_to_noisy(noisy_mag: torch.Tensor, noisy_phase: torch.Tensor, pred_mask: torch.Tensor, *,
                       use_log_mag: bool = True,
                       mask_activation: str = "tanh") -> torch.Tensor:
    """Apply complex ratio mask to noisy complex spectrum and return predicted complex spectrum.

    noisy_mag: (B, 1, F, T) or (B, F, T)
    noisy_phase: (B, F, T)
    pred_mask: (B, 2, F, T) -> [mask_real, mask_imag]
    """
    if pred_mask.dim() != 4 or pred_mask.size(1) != 2:
        raise ValueError(f"CRM expects model output shape (B,2,F,T), got {tuple(pred_mask.shape)}")

    if noisy_mag.dim() == 4:
        noisy_mag = noisy_mag.squeeze(1)

    if use_log_mag:
        noisy_mag = torch.expm1(noisy_mag)

    mask_real = pred_mask[:, 0]
    mask_imag = pred_mask[:, 1]
    if mask_activation == "tanh":
        mask_real = torch.tanh(mask_real)
        mask_imag = torch.tanh(mask_imag)

    noisy_complex = torch.polar(noisy_mag, noisy_phase)
    nr, ni = noisy_complex.real, noisy_complex.imag

    pred_real = mask_real * nr - mask_imag * ni
    pred_imag = mask_real * ni + mask_imag * nr

    return torch.complex(pred_real, pred_imag)


def build_target_complex(target_mag: torch.Tensor, target_phase: torch.Tensor, *, use_log_mag: bool = True) -> torch.Tensor:
    """Build target complex spectrum from magnitude + phase."""
    if target_mag.dim() == 4:
        target_mag = target_mag.squeeze(1)
    if use_log_mag:
        target_mag = torch.expm1(target_mag)
    return torch.polar(target_mag, target_phase)


def parse_args():
    parser = argparse.ArgumentParser(description="Train UNet denoiser")
    parser.add_argument('--loss', type=str, default=None, choices=['mse', 'l1', 'mrstft', 'hybrid', 'hybrid_l1', 'combined', 'crm'], help='Loss function to use (mse, l1, mrstft, hybrid, hybrid_l1, combined, crm)')
    parser.add_argument('--alpha', type=float, default=None, help='Alpha value for Hybrid Loss (weight for MR-STFT). Default is from config.')
    parser.add_argument('--lambda1', type=float, default=None, help='Lambda1 for Combined Loss (weight for MSE). Default is from config.')
    parser.add_argument('--lambda2', type=float, default=None, help='Lambda2 for Combined Loss (weight for MR-STFT). Default is from config.')
    parser.add_argument('--lr', type=float, default=None, help='Override initial learning rate (useful for fine-tuning)')
    parser.add_argument('--use-scheduler', dest='use_scheduler', action='store_true', default=cfg.USE_SCHEDULER, help='Enable Learning Rate Scheduler')
    parser.add_argument('--no-scheduler', dest='use_scheduler', action='store_false', help='Disable Learning Rate Scheduler')
    parser.add_argument('--reset-lr', action='store_true', help='Reset learning rate to initial value (ignore checkpoint LR)')
    parser.add_argument('--reset-best-loss', dest='reset_best_loss', action='store_true', help='Reset best_val_loss to inf when resuming (use when switching loss function/stage so the new stage can save unet_best.pt). Implicitly enabled when --reset-lr is set.')
    parser.add_argument('--dynamic', action='store_true', help='Use on-the-fly dynamic mixing instead of precomputed tensors')
    parser.add_argument('--mask-mode', action='store_true', default=False, help='Use ratio-mask prediction (sigmoid) instead of direct magnitude. Eliminates residual hiss.')
    parser.add_argument('--epochs', '--num_epochs', type=int, default=cfg.NUM_EPOCHS, help='Number of epochs to train. When resuming, treated as additional epochs (e.g. resume from 20 + --epochs 7 = train until epoch 27). When starting fresh, this is the total.')
    parser.add_argument('--resume', type=str, default=None, help='Path to checkpoint to resume from (loads model+optimizer state)')
    parser.add_argument('--start-epoch', dest='start_epoch', type=int, default=None, help='Override the starting epoch (useful when resuming best.pt but you want to count from the last trained epoch, e.g. --start-epoch 31)')
    parser.add_argument('--warmup-epochs', type=int, default=3, help='Number of LR warmup epochs (0 to disable warmup). Skipped when resuming with --reset-lr.')
    
    # Use parse_known_args to handle Jupyter/Colab kernel arguments (like -f ...) gracefully
    args, _ = parser.parse_known_args()
    return args





def get_dataloaders(root: str, cfg_obj, dynamic: bool = False):
    if dynamic:
        from dataset import DynamicDenoisingDataset
        from sklearn.model_selection import train_test_split
        import os

        def collect_files_internal(directory, extensions=(".wav", ".mp3", ".webm")):
            files = []
            if not os.path.exists(directory):
                return []
            for r, _, filenames in os.walk(directory):
                for f in filenames:
                    if f.lower().endswith(extensions):
                        files.append(os.path.join(r, f))
            return sorted(files)

        # Search across all potential raw directories
        clean_dirs = [cfg_obj.CLEAN_EN_DIR, cfg_obj.CLEAN_NP_DIR, cfg_obj.CLEAN_EN_MP3_DIR, cfg_obj.CLEAN_NP_WEBM_DIR]
        clean_files = []
        print(f"Searching for clean audio files in: {clean_dirs}")
        for d in clean_dirs:
            if not os.path.exists(d):
                print(f"  Warning: Directory not found: {d}")
                continue
            found = collect_files_internal(d)
            clean_files.extend(found)
            print(f"  Found {len(found)} files in {d}")

        noise_files = collect_files_internal(cfg_obj.NOISE_DIR)

        if not clean_files:
            raise ValueError(f"No clean files (.wav, .mp3, .webm) found in: {clean_dirs}")
        if not noise_files:
            raise ValueError(f"No noise files found in {cfg_obj.NOISE_DIR}")

        train_clean, val_clean = train_test_split(clean_files, test_size=0.2, random_state=42)
        
        target_mode = "noisy" if cfg_obj.NOISE2NOISE else "clean"
        train_ds = DynamicDenoisingDataset(train_clean, noise_files, split="train", return_phase=True, target=target_mode, cfg=cfg_obj)
        val_ds = DynamicDenoisingDataset(val_clean, noise_files, split="val", return_phase=True, target=target_mode, cfg=cfg_obj)
        print(f"Dynamic Mixing: {len(train_clean)} train, {len(val_clean)} val samples")
    else:
        target_mode = "noisy" if cfg_obj.NOISE2NOISE else "clean"
        # We now request phase to allow waveform reconstruction for MR-STFT
        train_ds = DenoisingDataset(root, split="train", target=target_mode, return_phase=True, cfg=cfg_obj)
        val_ds = DenoisingDataset(root, split="val", target=target_mode, return_phase=True, cfg=cfg_obj)
        print(f"Static Mode: {len(train_ds)} train, {len(val_ds)} val samples")


    common_loader_kwargs = dict(
        batch_size=cfg_obj.BATCH_SIZE,
        num_workers=cfg_obj.NUM_WORKERS,
        pin_memory=cfg_obj.PIN_MEMORY,
        persistent_workers=cfg_obj.NUM_WORKERS > 0,
        prefetch_factor=2 if cfg_obj.NUM_WORKERS > 0 else None,
    )

    train_loader = DataLoader(
        train_ds,
        shuffle=True,
        **{k: v for k, v in common_loader_kwargs.items() if v is not None},
    )

    val_loader = DataLoader(
        val_ds,
        shuffle=False,
        **{k: v for k, v in common_loader_kwargs.items() if v is not None},
    )
    return train_loader, val_loader


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    
    use_mrstft = False
    use_hybrid = False
    use_combined = False
    if MultiResolutionSTFTLoss is not None:
        use_mrstft = isinstance(criterion, MultiResolutionSTFTLoss)
    use_hybrid = isinstance(criterion, (HybridLoss, HybridL1Loss))
    use_crm = isinstance(criterion, CRMLoss)
    if CombinedLoss is not None:
        use_combined = isinstance(criterion, CombinedLoss)

    print(f"  Starting data loading for {len(loader)} batches...")
    optimizer.zero_grad()
    for batch_idx, batch_data in enumerate(loader):
        if batch_idx % 20 == 0:
            print(f"  Training batch {batch_idx}/{len(loader)}...", end='\r', flush=True)
            
        # Unpack based on return_phase=True/False
        if len(batch_data) == 4:
            noisy_input, target_mag, noisy_phase, target_phase = batch_data
            noisy_phase = noisy_phase.to(device)
            target_phase = target_phase.to(device)
        else:
            noisy_input, target_mag = batch_data
            noisy_phase, target_phase = None, None

        noisy_input = noisy_input.to(device)
        target_mag = target_mag.to(device)

        pred = model(noisy_input)
        
        if use_crm:
            if noisy_phase is None or target_phase is None:
                raise ValueError("CRM training requires both noisy_phase and target_phase")
            pred_complex = apply_crm_to_noisy(
                noisy_input,
                noisy_phase,
                pred,
                use_log_mag=cfg.USE_LOG_MAG,
                mask_activation=cfg.CRM_MASK_ACTIVATION,
            )
            target_complex = build_target_complex(target_mag, target_phase, use_log_mag=cfg.USE_LOG_MAG)
            loss = criterion(pred_complex, target_complex)
        elif use_hybrid:
            pred_wav = reconstruct_waveform_from_mag_and_phase(pred, noisy_phase)
            target_wav = reconstruct_waveform_from_mag_and_phase(target_mag, target_phase)
            loss = criterion(pred, target_mag, pred_wav, target_wav)
        elif use_combined:
            pred_wav = reconstruct_waveform_from_mag_and_phase(pred, noisy_phase)
            target_wav = reconstruct_waveform_from_mag_and_phase(target_mag, target_phase)
            # Returns combined_loss, mse, mrstft. We only backward the combined loss
            loss, mse_val, mrstft_val = criterion(pred_wav, target_wav)
        elif use_mrstft:
            # Reconstruct waveforms for time-domain loss
            # Use noisy_phase for prediction (standard approximation)
            pred_wav = reconstruct_waveform_from_mag_and_phase(pred, noisy_phase)
            # Use target_phase for target (ground truth waveform)
            target_wav = reconstruct_waveform_from_mag_and_phase(target_mag, target_phase)
            loss = criterion(pred_wav, target_wav)
        else:
            loss = criterion(pred, target_mag)
            
        # Scale loss for accumulation
        loss = loss / cfg.ACCUMULATION_STEPS
        loss.backward()
        
        if (batch_idx + 1) % cfg.ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(loader):
            optimizer.step()
            optimizer.zero_grad()

        running_loss += (loss.item() * cfg.ACCUMULATION_STEPS) * noisy_input.size(0)
    print(f"  Training batch {len(loader)}/{len(loader)} - Complete!        ")
    return running_loss / len(loader.dataset)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    
    use_mrstft = False
    use_hybrid = False
    use_combined = False
    if MultiResolutionSTFTLoss is not None:
        use_mrstft = isinstance(criterion, MultiResolutionSTFTLoss)
    use_hybrid = isinstance(criterion, (HybridLoss, HybridL1Loss))
    use_crm = isinstance(criterion, CRMLoss)
    if CombinedLoss is not None:
        use_combined = isinstance(criterion, CombinedLoss)
        
    for batch_data in loader:
        if len(batch_data) == 4:
            noisy_input, target_mag, noisy_phase, target_phase = batch_data
            noisy_phase = noisy_phase.to(device)
            target_phase = target_phase.to(device)
        else:
            noisy_input, target_mag = batch_data
            noisy_phase, target_phase = None, None

        noisy_input = noisy_input.to(device)
        target_mag = target_mag.to(device)

        pred = model(noisy_input)
        
        if use_crm:
            if noisy_phase is None or target_phase is None:
                raise ValueError("CRM validation requires both noisy_phase and target_phase")
            pred_complex = apply_crm_to_noisy(
                noisy_input,
                noisy_phase,
                pred,
                use_log_mag=cfg.USE_LOG_MAG,
                mask_activation=cfg.CRM_MASK_ACTIVATION,
            )
            target_complex = build_target_complex(target_mag, target_phase, use_log_mag=cfg.USE_LOG_MAG)
            loss = criterion(pred_complex, target_complex)
        elif use_hybrid:
            pred_wav = reconstruct_waveform_from_mag_and_phase(pred, noisy_phase)
            target_wav = reconstruct_waveform_from_mag_and_phase(target_mag, target_phase)
            loss = criterion(pred, target_mag, pred_wav, target_wav)
        elif use_combined:
            pred_wav = reconstruct_waveform_from_mag_and_phase(pred, noisy_phase)
            target_wav = reconstruct_waveform_from_mag_and_phase(target_mag, target_phase)
            loss, *_ = criterion(pred_wav, target_wav)
        elif use_mrstft:
            pred_wav = reconstruct_waveform_from_mag_and_phase(pred, noisy_phase)
            target_wav = reconstruct_waveform_from_mag_and_phase(target_mag, target_phase)
            loss = criterion(pred_wav, target_wav)
        else:
            loss = criterion(pred, target_mag)
            
        running_loss += loss.item() * noisy_input.size(0)
    return running_loss / len(loader.dataset)


def main():
    args = parse_args()

    # Build a per-run immutable config with any CLI overrides
    global cfg
    overrides = {}
    if args.loss is not None:
        overrides["LOSS_FUNCTION"] = args.loss
    if args.alpha is not None:
        overrides["HYBRID_ALPHA"] = args.alpha
    if args.lambda1 is not None:
        overrides["COMBINED_LAMBDA1"] = args.lambda1
    if args.lambda2 is not None:
        overrides["COMBINED_LAMBDA2"] = args.lambda2
    if args.lr is not None:
        overrides["INITIAL_LR"] = args.lr
    if args.use_scheduler != cfg.USE_SCHEDULER:
        overrides["USE_SCHEDULER"] = args.use_scheduler
    if args.epochs != cfg.NUM_EPOCHS:
        overrides["NUM_EPOCHS"] = args.epochs
    if args.mask_mode:
        overrides["MASK_MODE"] = True

    cfg = get_config(overrides if overrides else None)

    if torch.cuda.is_available():
        try:
            torch.backends.cuda.matmul.fp32_precision = "tf32"
        except Exception:
            pass
        try:
            torch.backends.cudnn.conv.fp32_precision = "tf32"
        except Exception:
            pass

    # Set up logging to the logs/ folder
    log_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "logs")
    logger = setup_logger(log_dir)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    data_root = cfg.PROCESSED_ROOT

    # Ensure checkpoint directory exists so we do not lose models after training
    checkpoint_dir = cfg.CHECKPOINT_DIR
    os.makedirs(checkpoint_dir, exist_ok=True)

    logger.info("Loading datasets...")
    train_loader, val_loader = get_dataloaders(data_root, cfg, dynamic=args.dynamic)
    logger.info(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

    # Logic for selecting the loss function
    # --loss argument overrides config
    loss_selection = cfg.LOSS_FUNCTION

    logger.info("Initializing model...")
    final_activation = "none" if loss_selection == "crm" else "softplus"
    use_mask_mode = cfg.MASK_MODE
    model = UNet(in_ch=cfg.UNET_IN_CH, out_ch=cfg.UNET_OUT_CH, base_ch=cfg.UNET_BASE_CH,
                 final_activation=final_activation, mask_mode=use_mask_mode).to(device)
    if use_mask_mode:
        logger.info("MASK MODE enabled — UNet predicts [0,1] ratio mask (sigmoid)")
    model.apply(init_kaiming)
    
    criterion = None
    use_mrstft_final = False

    if loss_selection == "hybrid":
        if MultiResolutionSTFTLoss is None:
            raise ImportError("MR_STFT module not found, cannot use hybrid loss")
        mrstft_loss = MultiResolutionSTFTLoss(
            fft_sizes=cfg.MR_STFT_FFT_SIZES,
            hop_sizes=cfg.MR_STFT_HOP_SIZES,
            win_lengths=cfg.MR_STFT_WIN_LENGTHS,
            window=cfg.MR_STFT_WINDOW
        ).to(device)
        
        alpha_val = cfg.HYBRID_ALPHA
        criterion = HybridLoss(mrstft_loss, alpha=alpha_val).to(device)
        logger.info(f"Using Hybrid Loss ({alpha_val} * MR-STFT + {1-alpha_val:.2f} * MSE)")

    elif loss_selection == "hybrid_l1":
        if MultiResolutionSTFTLoss is None:
            raise ImportError("MR_STFT module not found, cannot use hybrid_l1 loss")
        mrstft_loss = MultiResolutionSTFTLoss(
            fft_sizes=cfg.MR_STFT_FFT_SIZES,
            hop_sizes=cfg.MR_STFT_HOP_SIZES,
            win_lengths=cfg.MR_STFT_WIN_LENGTHS,
            window=cfg.MR_STFT_WINDOW
        ).to(device)
        alpha_val = cfg.HYBRID_ALPHA
        criterion = HybridL1Loss(mrstft_loss, alpha=alpha_val).to(device)
        logger.info(f"Using Hybrid L1 Loss ({alpha_val} * MR-STFT + {1-alpha_val:.2f} * L1) — robust to noisy targets")
            
    elif loss_selection == "combined":
        if CombinedLoss is None:
            raise ImportError("MR_STFT module not found, cannot use combined loss")
        
        l1 = cfg.COMBINED_LAMBDA1
        l2 = cfg.COMBINED_LAMBDA2
        logger.info(f"Using Combined Loss (L_total = {l1} * MSE + {l2} * MRSTFT)")
        
        criterion = CombinedLoss(
            lambda1=l1,
            lambda2=l2,
            fft_sizes=cfg.MR_STFT_FFT_SIZES,
            hop_sizes=cfg.MR_STFT_HOP_SIZES,
            win_lengths=cfg.MR_STFT_WIN_LENGTHS,
            window=cfg.MR_STFT_WINDOW
        ).to(device)

    elif loss_selection == "mrstft":
        if MultiResolutionSTFTLoss is None:
            raise ImportError("MR_STFT module not found, cannot use mrstft loss")
        criterion = MultiResolutionSTFTLoss(
            fft_sizes=cfg.MR_STFT_FFT_SIZES,
            hop_sizes=cfg.MR_STFT_HOP_SIZES,
            win_lengths=cfg.MR_STFT_WIN_LENGTHS,
            window=cfg.MR_STFT_WINDOW
        ).to(device)
        logger.info("Using Multi-Resolution STFT Loss (MR-STFT)")

    elif loss_selection == "crm":
        if cfg.UNET_OUT_CH != 2:
            raise ValueError("CRM requires UNET_OUT_CH=2 (real and imaginary mask channels)")
        criterion = CRMLoss().to(device)
        logger.info(f"Using CRM Loss (mask activation: {cfg.CRM_MASK_ACTIVATION})")

    elif loss_selection == "l1":
        criterion = nn.L1Loss().to(device)
        logger.info("Using L1 Loss (median-seeking, robust to noisy targets)")

    else:
        # Default to MSE
        criterion = nn.MSELoss().to(device)
        logger.info("Using MSE Loss")
        
    optimizer = None
    initial_lr = cfg.INITIAL_LR
    logger.info(f"Initial Learning Rate: {initial_lr}")
    wd = cfg.WEIGHT_DECAY if cfg.USE_WEIGHT_DECAY else 0.0
    
    if hasattr(torch.optim, "AdamW"):
        fused_ok = torch.cuda.is_available() and hasattr(torch.optim.AdamW, "fused")
        try:
            optimizer = torch.optim.AdamW(model.parameters(), lr=initial_lr, betas=cfg.ADAM_BETAS, weight_decay=wd, fused=fused_ok)
            logger.info("Using fused AdamW optimizer" if fused_ok else "Using AdamW optimizer")
        except TypeError:
            optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr, weight_decay=wd)
            logger.info("Fused AdamW unavailable; using Adam")
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr, weight_decay=wd)
        logger.info("AdamW not found; using Adam")

    # -- Resume checkpoint first (so we know start_epoch for scheduler) --
    start_epoch = 1
    best_val_loss = float("inf")
    if args.resume is not None:
        if os.path.isfile(args.resume):
            logger.info(f"Resuming from checkpoint: {args.resume}")
            ckpt = torch.load(args.resume, map_location=device)
            state_dict = ckpt.get("model_state_dict") if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
            model.load_state_dict(state_dict)
            
            if isinstance(ckpt, dict) and "optimizer_state_dict" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                
            if isinstance(ckpt, dict) and "epoch" in ckpt:
                start_epoch = ckpt["epoch"] + 1
            if isinstance(ckpt, dict) and "best_val_loss" in ckpt:
                best_val_loss = ckpt["best_val_loss"]

            # Reset best_val_loss when starting a new stage (different loss function scale)
            # Triggered explicitly via --reset-best-loss or implicitly via --reset-lr
            if args.reset_best_loss or args.reset_lr:
                logger.info(f"Resetting best_val_loss from {best_val_loss:.4f} → inf (new stage, loss scale may differ)")
                best_val_loss = float("inf")

            # FORCED LR RESET Logic
            if args.reset_lr or args.lr is not None:
                new_lr = args.lr if args.lr is not None else initial_lr
                for param_group in optimizer.param_groups:
                    param_group['lr'] = new_lr
                logger.warning(f"FORCING LEARNING RATE TO {new_lr}")

            logger.info(f"Resumed at epoch {start_epoch} with best_val_loss={best_val_loss:.4f}")
        else:
            logger.warning(f"Resume checkpoint not found at {args.resume}; starting fresh")

    # Allow manual override of start epoch (e.g. resume best.pt but continue from epoch 31)
    if args.start_epoch is not None:
        logger.info(f"Overriding start_epoch: {start_epoch} → {args.start_epoch} (via --start-epoch)")
        start_epoch = args.start_epoch

    # -- Build scheduler AFTER resume so it knows start_epoch --
    is_resuming = (args.resume is not None and start_epoch > 1)

    # Treat --epochs as *additional* epochs when resuming.
    # e.g. resume from epoch 20, --epochs 7 → train until epoch 27
    num_epochs = (start_epoch - 1 + args.epochs) if is_resuming else args.epochs

    scheduler = None
    if args.use_scheduler:
        import math as _math
        # When resuming with --reset-lr, skip warmup (already trained)
        warmup_epochs = 0 if (is_resuming and args.reset_lr) else args.warmup_epochs
        # Cosine annealing spans from start_epoch to num_epochs
        # so the LR decays smoothly over the REMAINING training, not the full run
        effective_start = start_epoch if is_resuming else 0
        effective_total = max(1, num_epochs - effective_start)
        min_lr_ratio = 1e-6 / initial_lr

        def lr_lambda(epoch):
            # epoch is 0-indexed within the scheduler (calls since creation)
            # Map to actual training progress
            actual_epoch = effective_start + epoch
            if epoch < warmup_epochs:
                # Linear warmup from 10% to 100%
                return 0.1 + 0.9 * float(epoch) / float(max(1, warmup_epochs))
            else:
                # Cosine annealing from 1 to min_lr_ratio
                progress_after_warmup = float(epoch - warmup_epochs) / float(max(1, effective_total - warmup_epochs))
                progress_after_warmup = min(progress_after_warmup, 1.0)
                return min_lr_ratio + 0.5 * (1.0 - min_lr_ratio) * (1.0 + _math.cos(_math.pi * progress_after_warmup))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        if warmup_epochs > 0:
            logger.info(f"Scheduler: Linear Warmup ({warmup_epochs} epochs) + Cosine Annealing over {effective_total} epochs")
        else:
            logger.info(f"Scheduler: Cosine Annealing over {effective_total} remaining epochs (warmup skipped — resume with reset-lr)")
    else:
        logger.info("Learning Rate Scheduler DISABLED")

    if is_resuming:
        logger.info(f"Resuming from epoch {start_epoch - 1}. Running {args.epochs} additional epochs → training until epoch {num_epochs}.")
    else:
        logger.info(f"Starting training for {num_epochs} epochs...")
    epoch = start_epoch - 1  # initialize for interruption safety
    try:
        for epoch in range(start_epoch, num_epochs + 1):
            current_lr = optimizer.param_groups[0]['lr']
            logger.info(f"Epoch {epoch}/{num_epochs} | LR: {current_lr:.2e}")
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_loss = validate(model, val_loader, criterion, device)
            logger.info(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

            # Update learning rate (LambdaLR steps on epoch, not on loss like ReduceLROnPlateau)
            if scheduler is not None:
                scheduler.step()

            # Save checkpoint for every epoch
            ckpt_payload = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "best_val_loss": best_val_loss,
            }
            ckpt_path = os.path.join(checkpoint_dir, f"unet_epoch_{epoch:02d}.pt")
            torch.save(ckpt_payload, ckpt_path)

            # Track the best validation loss and keep a convenient copy
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                ckpt_payload["best_val_loss"] = best_val_loss
                best_ckpt_path = os.path.join(checkpoint_dir, "unet_best.pt")
                torch.save(ckpt_payload, best_ckpt_path)
                logger.info(f"New best checkpoint saved to {best_ckpt_path}")
    except KeyboardInterrupt:
        paused_path = os.path.join(checkpoint_dir, "unet_paused.pt")
        logger.warning("Training interrupted. Saving paused checkpoint...")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "best_val_loss": best_val_loss,
        }, paused_path)
        logger.info(f"Paused checkpoint saved to {paused_path}")
        return

    logger.info("Saving final model state dict...")
    torch.save(model.state_dict(), os.path.join(checkpoint_dir, "unet_placeholder.pt"))
    logger.info(f"Training complete! Final model saved to {os.path.join(checkpoint_dir, 'unet_placeholder.pt')}")


if __name__ == "__main__":
    
    main()



Overwriting train.py


In [ ]:
%%writefile evalutate.py
import argparse
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from config import (
    BATCH_SIZE, NUM_WORKERS, PIN_MEMORY, SAMPLE_RATE, PROCESSED_ROOT, NOISE2NOISE, RESULTS_DIR, CHECKPOINT_DIR,
    UNET_IN_CH, UNET_OUT_CH, UNET_BASE_CH, CLEAN_EN_DIR, NOISE_DIR,
    PREFER_INPUT_PHASE_RECON, APPLY_POSTFILTER, MASK_MODE
)
from dataset import DenoisingDataset, DynamicDenoisingDataset
from utilis import reconstruct_waveform_from_mag_and_phase, reconstruct_waveform_auto
from model import UNet

try:
    import soundfile as sf

    _HAS_SF = True
except Exception:
    _HAS_SF = False

try:
    from pystoi import stoi

    _HAS_STOI = True
except Exception:
    _HAS_STOI = False


def psnr_per_sample(clean: torch.Tensor, pred: torch.Tensor) -> torch.Tensor:
    """Compute PSNR per sample. Assumes tensors of shape (B, C, H, W).
    Uses per-sample max of absolute clean signal as peak reference.
    """
    b = clean.size(0)
    clean_flat = clean.view(b, -1)
    pred_flat = pred.view(b, -1)
    mse = torch.mean((clean_flat - pred_flat) ** 2, dim=1)
    max_val = clean_flat.abs().amax(dim=1)
    # avoid division by zero
    mse = torch.clamp(mse, min=1e-12)
    max_val = torch.clamp(max_val, min=1e-12)
    psnr = 10.0 * torch.log10((max_val ** 2) / mse)
    return psnr


def snr_per_sample(clean: torch.Tensor, pred: torch.Tensor) -> torch.Tensor:
    b = clean.size(0)
    clean_flat = clean.view(b, -1)
    pred_flat = pred.view(b, -1)
    signal_power = torch.sum(clean_flat ** 2, dim=1)
    noise_power = torch.sum((clean_flat - pred_flat) ** 2, dim=1)
    
    # Avoid log(0) for extremely low signal power (silence)
    signal_power = torch.clamp(signal_power, min=1e-12)
    noise_power = torch.clamp(noise_power, min=1e-12)
    
    snr = 10.0 * torch.log10(signal_power / noise_power)
    return snr


def mse_l1_per_sample(clean: torch.Tensor, pred: torch.Tensor):
    b = clean.size(0)
    clean_flat = clean.view(b, -1)
    pred_flat = pred.view(b, -1)
    mse = torch.mean((clean_flat - pred_flat) ** 2, dim=1)
    l1 = torch.mean(torch.abs(clean_flat - pred_flat), dim=1)
    return mse, l1


def lsd_per_sample(clean: torch.Tensor, pred: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Log-spectral distance over frequency, averaged over time."""

    clean_mag = clean.squeeze(1)
    pred_mag = pred.squeeze(1)
    clean_log = torch.log10(clean_mag + eps)
    pred_log = torch.log10(pred_mag + eps)
    diff2 = (clean_log - pred_log) ** 2
    # mean over freq then sqrt per frame, then mean over time -> per sample
    per_frame = torch.sqrt(diff2.mean(dim=1))
    return per_frame.mean(dim=1)


def _gaussian_window(window_size: int = 11, sigma: float = 1.5) -> torch.Tensor:
    coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = torch.outer(g, g)
    return window_2d.unsqueeze(0).unsqueeze(0)


def ssim_per_sample(clean: torch.Tensor, pred: torch.Tensor, window: torch.Tensor = None, window_size: int = 11) -> torch.Tensor:
    """Compute SSIM per sample for spectrogram-like tensors (B,1,F,T)."""

    if window is None or window.device != clean.device or window.dtype != clean.dtype:
        window = _gaussian_window(window_size=window_size).to(device=clean.device, dtype=clean.dtype)
    pad = window_size // 2

    mu_x = F.conv2d(clean, window, padding=pad, groups=1)
    mu_y = F.conv2d(pred, window, padding=pad, groups=1)

    mu_x2 = mu_x ** 2
    mu_y2 = mu_y ** 2
    mu_xy = mu_x * mu_y

    sigma_x2 = F.conv2d(clean * clean, window, padding=pad, groups=1) - mu_x2
    sigma_y2 = F.conv2d(pred * pred, window, padding=pad, groups=1) - mu_y2
    sigma_xy = F.conv2d(clean * pred, window, padding=pad, groups=1) - mu_xy

    data_range = (torch.max(torch.stack([clean.max(), pred.max()])) - torch.min(torch.stack([clean.min(), pred.min()]))).clamp(min=1e-4)
    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    ssim_map = ((2 * mu_xy + C1) * (2 * sigma_xy + C2)) / ((mu_x2 + mu_y2 + C1) * (sigma_x2 + sigma_y2 + C2))
    b = clean.size(0)
    return ssim_map.view(b, -1).mean(dim=1)


def get_val_loader(root: str, split: str = "val", return_phase: bool = False, target: str = "clean", dynamic: bool = False):
    if dynamic:
        # For dynamic evaluation, we load raw audio files directly
        # Note: clean_en is just one example; you might want to evaluate on clean_np, etc.
        # or combine them. Here, we stick to CLEAN_EN_DIR for consistency with training demo.
        clean_files = sorted([os.path.join(CLEAN_EN_DIR, f) for f in os.listdir(CLEAN_EN_DIR) if f.endswith(".wav")])
        
        # Collect all noise files
        noise_files = []
        if os.path.exists(NOISE_DIR):
             for root, dirs, files in os.walk(NOISE_DIR):
                for f in files:
                    if f.endswith(".wav"):
                        noise_files.append(os.path.join(root, f))
        
        val_ds = DynamicDenoisingDataset(clean_files, noise_files, split=split, return_phase=return_phase, target=target)
    else:
        val_ds = DenoisingDataset(root, split=split, return_phase=return_phase, target=target)
    
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    return val_loader


@torch.no_grad()
def evaluate(
    checkpoint: str,
    data_root: str = PROCESSED_ROOT,
    split: str = "val",
    reconstruct: bool = False,
    out_dir: str = f"{RESULTS_DIR}/recon",
    enable_stoi: bool = True,
    compare_to_clean: bool = True,
    dynamic: bool = False,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Check if CUDA is actually usable despite being "available"
    if device.type == "cuda":
        try:
            # Check capabilities
            cap = torch.cuda.get_device_capability(device)
            if cap[0] < 7 or (cap[0] == 7 and cap[1] < 5):
                 print(f"WARNING: Detected GPU with capability {cap[0]}.{cap[1]}. This PyTorch version may require 7.5+.")
                 print("Examples of 6.1 GPUs (unsupported by this PT binary): GTX 1050/1060/1070/1080.")
                 print("Continuing with CUDA enabled (may crash if unsupported ops are used).")
        except Exception:
            pass

    print(f"Using device: {device}")

    model = UNet(in_ch=UNET_IN_CH, out_ch=UNET_OUT_CH, base_ch=UNET_BASE_CH, mask_mode=MASK_MODE).to(device)
    state = torch.load(checkpoint, map_location=device)
    # Support both raw state_dict (old behavior) and full checkpoint dict
    state_dict = state.get("model_state_dict") if isinstance(state, dict) and "model_state_dict" in state else state
    model.load_state_dict(state_dict)
    model.eval()

    # If clean targets are available, keep compare_to_clean=True (recommended for metrics).
    # If evaluating strictly Noise2Noise targets, set compare_to_clean=False to compare against noisy-B.
    target_mode = "clean" if compare_to_clean else "noisy"
    need_phase = reconstruct or (enable_stoi and _HAS_STOI)
    val_loader = get_val_loader(data_root, split=split, return_phase=need_phase, target=target_mode, dynamic=dynamic)
    all_psnr = []
    all_snr = []
    all_mse = []
    all_l1 = []
    all_lsd = []
    all_ssim = []
    all_stoi = []
    file_counter = 0
    num_processed = 0
    window = None

    for batch in val_loader:
        if need_phase:
            # target_mode dictates what phases come back
            if target_mode == "clean":
                noisy, target_mag, noisy_phase, clean_phase = batch
            else:
                noisy, target_mag, noisy_phase, _ = batch  # second phase unused here
                clean_phase = None
        else:
            noisy, target_mag = batch
            noisy_phase = None
            clean_phase = None

        noisy = noisy.to(device)
        target_mag = target_mag.to(device)
        pred = model(noisy)
        pred = pred.type_as(target_mag)

        psnr_vals = psnr_per_sample(target_mag, pred)
        snr_vals = snr_per_sample(target_mag, pred)
        mse_vals, l1_vals = mse_l1_per_sample(target_mag, pred)
        lsd_vals = lsd_per_sample(target_mag, pred)
        if window is None or window.device != pred.device or window.dtype != pred.dtype:
            window = _gaussian_window().to(device=pred.device, dtype=pred.dtype)
        ssim_vals = ssim_per_sample(target_mag, pred, window=window)

        all_psnr.append(psnr_vals.cpu())
        all_snr.append(snr_vals.cpu())
        all_mse.append(mse_vals.cpu())
        all_l1.append(l1_vals.cpu())
        all_lsd.append(lsd_vals.cpu())
        all_ssim.append(ssim_vals.cpu())

        num_processed += noisy.size(0)
        if num_processed % 4 == 0:  # Print frequently since CPU is slow
             print(f"Processed {num_processed} samples...", end='\r', flush=True)

        if compare_to_clean and enable_stoi and _HAS_STOI and noisy_phase is not None and clean_phase is not None:
            pred_mag = pred.squeeze(1).detach().cpu()
            noisy_phase_cpu = noisy_phase.cpu()
            clean_mag = target_mag.squeeze(1).detach().cpu()
            clean_phase_cpu = clean_phase.cpu()

            try:
                # Pass noisy magnitude (input) as reference for Nyquist bin
                # The `noisy` variable holds the input magnitude
                pred_wav = reconstruct_waveform_from_mag_and_phase(
                    pred_mag, 
                    noisy_phase_cpu,
                    ref_mag=noisy.cpu()
                )
                clean_wav = reconstruct_waveform_from_mag_and_phase(clean_mag, clean_phase_cpu)

                stoi_scores = []
                for pw, cw in zip(pred_wav, clean_wav):
                    min_len = min(pw.numel(), cw.numel())
                    if min_len == 0:
                        stoi_scores.append(torch.tensor(0.0))
                        continue
                    score = stoi(
                        cw[:min_len].numpy(),
                        pw[:min_len].numpy(),
                        SAMPLE_RATE,
                        extended=False,
                    )
                    stoi_scores.append(torch.tensor(float(score)))

                if len(stoi_scores) > 0:
                    all_stoi.append(torch.stack(stoi_scores))
            except Exception:
                # If reconstruction fails, skip STOI for this batch
                pass

        if reconstruct:
            pred_mag = pred.squeeze(1).cpu()
            noisy_phase_cpu = noisy_phase.cpu() if noisy_phase is not None else None
            # Pass original noisy magnitude (target_mag actually usually is clean/target, we need noisy for reference)
            # But the loop loads noisy as (B, 1, F, T) -> noisy.
            noisy_mag_cpu = noisy.cpu()
            
            wave_batch = reconstruct_waveform_auto(
                pred_mag,
                noisy_phase_cpu,
                ref_mag=noisy_mag_cpu,
                prefer_input_phase=PREFER_INPUT_PHASE_RECON,
                apply_postfilter=APPLY_POSTFILTER,
            )
            os.makedirs(out_dir, exist_ok=True)
            for i, waveform in enumerate(wave_batch):
                path = os.path.join(out_dir, f"recon_{file_counter:08d}.wav")
                wav_np = waveform.detach().numpy()
                if _HAS_SF:
                    sf.write(path, wav_np, samplerate=SAMPLE_RATE)
                file_counter += 1

    print("") # Newline
    all_psnr = torch.cat(all_psnr)
    all_snr = torch.cat(all_snr)
    all_mse = torch.cat(all_mse)
    all_l1 = torch.cat(all_l1)
    all_lsd = torch.cat(all_lsd)
    all_ssim = torch.cat(all_ssim)
    all_stoi_cat = torch.cat(all_stoi) if len(all_stoi) > 0 else None

    print(f"Validation samples: {all_psnr.numel()}")
    print(f"Average PSNR: {all_psnr.mean().item():.3f} dB")
    print(f"Median PSNR: {all_psnr.median().item():.3f} dB")
    print(f"Average SNR: {all_snr.mean().item():.3f} dB")
    print(f"Median SNR: {all_snr.median().item():.3f} dB")
    print(f"Average MSE: {all_mse.mean().item():.6f}")
    print(f"Average L1: {all_l1.mean().item():.6f}")
    print(f"Average LSD: {all_lsd.mean().item():.6f}")
    print(f"Average SSIM: {all_ssim.mean().item():.6f}")
    if enable_stoi:
        if _HAS_STOI and all_stoi_cat is not None:
            print(f"Average STOI: {all_stoi_cat.mean().item():.6f}")
        elif not _HAS_STOI:
            print("STOI skipped: install `pystoi` to enable.")
        else:
            print("STOI skipped: phase or reconstruction unavailable.")


def parse_args():
    p = argparse.ArgumentParser(description="Evaluate UNet denoiser on validation or test set")
    # Use config paths as defaults
    ckpt_default = os.path.join(CHECKPOINT_DIR, "unet_placeholder.pt")
    
    p.add_argument("--checkpoint", default=ckpt_default, help="Path to model checkpoint")
    p.add_argument("--data-root", default=PROCESSED_ROOT, help="Processed dataset root")
    p.add_argument("--split", default="val", choices=["val", "test"], help="Dataset split to evaluate (val or test)")
    p.add_argument("--reconstruct", action="store_true", help="Reconstruct waveforms using noisy phase and save to --out-dir")
    p.add_argument("--out-dir", default=f"{RESULTS_DIR}/recon", help="Output directory for reconstructed WAVs")
    p.add_argument("--no-stoi", action="store_true", help="Skip STOI computation even if pystoi is installed")
    p.add_argument("--dynamic", action="store_true", help="Use dynamic dataset generation (on-the-fly mixing)")
    
    # Use parse_known_args to handle Jupyter/Colab kernel arguments (like -f ...) gracefully
    args, _ = p.parse_known_args()
    return args

if __name__ == "__main__":
    args = parse_args()
    evaluate(
        args.checkpoint,
        args.data_root,
        split=args.split,
        reconstruct=args.reconstruct,
        out_dir=args.out_dir,
        enable_stoi=not args.no_stoi,
        dynamic=args.dynamic,
    )

In [ ]:
%%writefile visualize.py
#visualize.py

import argparse
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from config import (
    BATCH_SIZE, NUM_WORKERS, PIN_MEMORY, SAMPLE_RATE, PROCESSED_ROOT, NOISE2NOISE, RESULTS_DIR, CHECKPOINT_DIR,
    UNET_IN_CH, UNET_OUT_CH, UNET_BASE_CH, CLEAN_EN_DIR, NOISE_DIR,
    PREFER_INPUT_PHASE_RECON, APPLY_POSTFILTER, MASK_MODE
)
from dataset import DenoisingDataset, DynamicDenoisingDataset
from utilis import reconstruct_waveform_from_mag_and_phase, reconstruct_waveform_auto
from model import UNet

try:
    import soundfile as sf

    _HAS_SF = True
except Exception:
    _HAS_SF = False

try:
    from pystoi import stoi

    _HAS_STOI = True
except Exception:
    _HAS_STOI = False


def psnr_per_sample(clean: torch.Tensor, pred: torch.Tensor) -> torch.Tensor:
    """Compute PSNR per sample. Assumes tensors of shape (B, C, H, W).
    Uses per-sample max of absolute clean signal as peak reference.
    """
    b = clean.size(0)
    clean_flat = clean.view(b, -1)
    pred_flat = pred.view(b, -1)
    mse = torch.mean((clean_flat - pred_flat) ** 2, dim=1)
    max_val = clean_flat.abs().amax(dim=1)
    # avoid division by zero
    mse = torch.clamp(mse, min=1e-12)
    max_val = torch.clamp(max_val, min=1e-12)
    psnr = 10.0 * torch.log10((max_val ** 2) / mse)
    return psnr


def snr_per_sample(clean: torch.Tensor, pred: torch.Tensor) -> torch.Tensor:
    b = clean.size(0)
    clean_flat = clean.view(b, -1)
    pred_flat = pred.view(b, -1)
    signal_power = torch.sum(clean_flat ** 2, dim=1)
    noise_power = torch.sum((clean_flat - pred_flat) ** 2, dim=1)
    
    # Avoid log(0) for extremely low signal power (silence)
    signal_power = torch.clamp(signal_power, min=1e-12)
    noise_power = torch.clamp(noise_power, min=1e-12)
    
    snr = 10.0 * torch.log10(signal_power / noise_power)
    return snr


def mse_l1_per_sample(clean: torch.Tensor, pred: torch.Tensor):
    b = clean.size(0)
    clean_flat = clean.view(b, -1)
    pred_flat = pred.view(b, -1)
    mse = torch.mean((clean_flat - pred_flat) ** 2, dim=1)
    l1 = torch.mean(torch.abs(clean_flat - pred_flat), dim=1)
    return mse, l1


def lsd_per_sample(clean: torch.Tensor, pred: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Log-spectral distance over frequency, averaged over time."""

    clean_mag = clean.squeeze(1)
    pred_mag = pred.squeeze(1)
    clean_log = torch.log10(clean_mag + eps)
    pred_log = torch.log10(pred_mag + eps)
    diff2 = (clean_log - pred_log) ** 2
    # mean over freq then sqrt per frame, then mean over time -> per sample
    per_frame = torch.sqrt(diff2.mean(dim=1))
    return per_frame.mean(dim=1)


def _gaussian_window(window_size: int = 11, sigma: float = 1.5) -> torch.Tensor:
    coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = torch.outer(g, g)
    return window_2d.unsqueeze(0).unsqueeze(0)


def ssim_per_sample(clean: torch.Tensor, pred: torch.Tensor, window: torch.Tensor = None, window_size: int = 11) -> torch.Tensor:
    """Compute SSIM per sample for spectrogram-like tensors (B,1,F,T)."""

    if window is None or window.device != clean.device or window.dtype != clean.dtype:
        window = _gaussian_window(window_size=window_size).to(device=clean.device, dtype=clean.dtype)
    pad = window_size // 2

    mu_x = F.conv2d(clean, window, padding=pad, groups=1)
    mu_y = F.conv2d(pred, window, padding=pad, groups=1)

    mu_x2 = mu_x ** 2
    mu_y2 = mu_y ** 2
    mu_xy = mu_x * mu_y

    sigma_x2 = F.conv2d(clean * clean, window, padding=pad, groups=1) - mu_x2
    sigma_y2 = F.conv2d(pred * pred, window, padding=pad, groups=1) - mu_y2
    sigma_xy = F.conv2d(clean * pred, window, padding=pad, groups=1) - mu_xy

    data_range = (torch.max(torch.stack([clean.max(), pred.max()])) - torch.min(torch.stack([clean.min(), pred.min()]))).clamp(min=1e-4)
    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    ssim_map = ((2 * mu_xy + C1) * (2 * sigma_xy + C2)) / ((mu_x2 + mu_y2 + C1) * (sigma_x2 + sigma_y2 + C2))
    b = clean.size(0)
    return ssim_map.view(b, -1).mean(dim=1)


def get_val_loader(root: str, split: str = "val", return_phase: bool = False, target: str = "clean", dynamic: bool = False):
    if dynamic:
        # For dynamic evaluation, we load raw audio files directly
        # Note: clean_en is just one example; you might want to evaluate on clean_np, etc.
        # or combine them. Here, we stick to CLEAN_EN_DIR for consistency with training demo.
        clean_files = sorted([os.path.join(CLEAN_EN_DIR, f) for f in os.listdir(CLEAN_EN_DIR) if f.endswith(".wav")])
        
        # Collect all noise files
        noise_files = []
        if os.path.exists(NOISE_DIR):
             for root, dirs, files in os.walk(NOISE_DIR):
                for f in files:
                    if f.endswith(".wav"):
                        noise_files.append(os.path.join(root, f))
        
        val_ds = DynamicDenoisingDataset(clean_files, noise_files, split=split, return_phase=return_phase, target=target)
    else:
        val_ds = DenoisingDataset(root, split=split, return_phase=return_phase, target=target)
    
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    return val_loader


@torch.no_grad()
def evaluate(
    checkpoint: str,
    data_root: str = PROCESSED_ROOT,
    split: str = "val",
    reconstruct: bool = False,
    out_dir: str = f"{RESULTS_DIR}/recon",
    enable_stoi: bool = True,
    compare_to_clean: bool = True,
    dynamic: bool = False,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Check if CUDA is actually usable despite being "available"
    if device.type == "cuda":
        try:
            # Check capabilities
            cap = torch.cuda.get_device_capability(device)
            if cap[0] < 7 or (cap[0] == 7 and cap[1] < 5):
                 print(f"WARNING: Detected GPU with capability {cap[0]}.{cap[1]}. This PyTorch version may require 7.5+.")
                 print("Examples of 6.1 GPUs (unsupported by this PT binary): GTX 1050/1060/1070/1080.")
                 print("Continuing with CUDA enabled (may crash if unsupported ops are used).")
        except Exception:
            pass

    print(f"Using device: {device}")

    model = UNet(in_ch=UNET_IN_CH, out_ch=UNET_OUT_CH, base_ch=UNET_BASE_CH, mask_mode=MASK_MODE).to(device)
    state = torch.load(checkpoint, map_location=device)
    # Support both raw state_dict (old behavior) and full checkpoint dict
    state_dict = state.get("model_state_dict") if isinstance(state, dict) and "model_state_dict" in state else state
    model.load_state_dict(state_dict)
    model.eval()

    # If clean targets are available, keep compare_to_clean=True (recommended for metrics).
    # If evaluating strictly Noise2Noise targets, set compare_to_clean=False to compare against noisy-B.
    target_mode = "clean" if compare_to_clean else "noisy"
    need_phase = reconstruct or (enable_stoi and _HAS_STOI)
    val_loader = get_val_loader(data_root, split=split, return_phase=need_phase, target=target_mode, dynamic=dynamic)
    all_psnr = []
    all_snr = []
    all_mse = []
    all_l1 = []
    all_lsd = []
    all_ssim = []
    all_stoi = []
    file_counter = 0
    num_processed = 0
    window = None

    for batch in val_loader:
        if need_phase:
            # target_mode dictates what phases come back
            if target_mode == "clean":
                noisy, target_mag, noisy_phase, clean_phase = batch
            else:
                noisy, target_mag, noisy_phase, _ = batch  # second phase unused here
                clean_phase = None
        else:
            noisy, target_mag = batch
            noisy_phase = None
            clean_phase = None

        noisy = noisy.to(device)
        target_mag = target_mag.to(device)
        pred = model(noisy)
        pred = pred.type_as(target_mag)

        psnr_vals = psnr_per_sample(target_mag, pred)
        snr_vals = snr_per_sample(target_mag, pred)
        mse_vals, l1_vals = mse_l1_per_sample(target_mag, pred)
        lsd_vals = lsd_per_sample(target_mag, pred)
        if window is None or window.device != pred.device or window.dtype != pred.dtype:
            window = _gaussian_window().to(device=pred.device, dtype=pred.dtype)
        ssim_vals = ssim_per_sample(target_mag, pred, window=window)

        all_psnr.append(psnr_vals.cpu())
        all_snr.append(snr_vals.cpu())
        all_mse.append(mse_vals.cpu())
        all_l1.append(l1_vals.cpu())
        all_lsd.append(lsd_vals.cpu())
        all_ssim.append(ssim_vals.cpu())

        num_processed += noisy.size(0)
        if num_processed % 4 == 0:  # Print frequently since CPU is slow
             print(f"Processed {num_processed} samples...", end='\r', flush=True)

        if compare_to_clean and enable_stoi and _HAS_STOI and noisy_phase is not None and clean_phase is not None:
            pred_mag = pred.squeeze(1).detach().cpu()
            noisy_phase_cpu = noisy_phase.cpu()
            clean_mag = target_mag.squeeze(1).detach().cpu()
            clean_phase_cpu = clean_phase.cpu()

            try:
                # Pass noisy magnitude (input) as reference for Nyquist bin
                # The `noisy` variable holds the input magnitude
                pred_wav = reconstruct_waveform_from_mag_and_phase(
                    pred_mag, 
                    noisy_phase_cpu,
                    ref_mag=noisy.cpu()
                )
                clean_wav = reconstruct_waveform_from_mag_and_phase(clean_mag, clean_phase_cpu)

                stoi_scores = []
                for pw, cw in zip(pred_wav, clean_wav):
                    min_len = min(pw.numel(), cw.numel())
                    if min_len == 0:
                        stoi_scores.append(torch.tensor(0.0))
                        continue
                    score = stoi(
                        cw[:min_len].numpy(),
                        pw[:min_len].numpy(),
                        SAMPLE_RATE,
                        extended=False,
                    )
                    stoi_scores.append(torch.tensor(float(score)))

                if len(stoi_scores) > 0:
                    all_stoi.append(torch.stack(stoi_scores))
            except Exception:
                # If reconstruction fails, skip STOI for this batch
                pass

        if reconstruct:
            pred_mag = pred.squeeze(1).cpu()
            noisy_phase_cpu = noisy_phase.cpu() if noisy_phase is not None else None
            # Pass original noisy magnitude (target_mag actually usually is clean/target, we need noisy for reference)
            # But the loop loads noisy as (B, 1, F, T) -> noisy.
            noisy_mag_cpu = noisy.cpu()
            
            wave_batch = reconstruct_waveform_auto(
                pred_mag,
                noisy_phase_cpu,
                ref_mag=noisy_mag_cpu,
                prefer_input_phase=PREFER_INPUT_PHASE_RECON,
                apply_postfilter=APPLY_POSTFILTER,
            )
            os.makedirs(out_dir, exist_ok=True)
            for i, waveform in enumerate(wave_batch):
                path = os.path.join(out_dir, f"recon_{file_counter:08d}.wav")
                wav_np = waveform.detach().numpy()
                if _HAS_SF:
                    sf.write(path, wav_np, samplerate=SAMPLE_RATE)
                file_counter += 1

    print("") # Newline
    all_psnr = torch.cat(all_psnr)
    all_snr = torch.cat(all_snr)
    all_mse = torch.cat(all_mse)
    all_l1 = torch.cat(all_l1)
    all_lsd = torch.cat(all_lsd)
    all_ssim = torch.cat(all_ssim)
    all_stoi_cat = torch.cat(all_stoi) if len(all_stoi) > 0 else None

    print(f"Validation samples: {all_psnr.numel()}")
    print(f"Average PSNR: {all_psnr.mean().item():.3f} dB")
    print(f"Median PSNR: {all_psnr.median().item():.3f} dB")
    print(f"Average SNR: {all_snr.mean().item():.3f} dB")
    print(f"Median SNR: {all_snr.median().item():.3f} dB")
    print(f"Average MSE: {all_mse.mean().item():.6f}")
    print(f"Average L1: {all_l1.mean().item():.6f}")
    print(f"Average LSD: {all_lsd.mean().item():.6f}")
    print(f"Average SSIM: {all_ssim.mean().item():.6f}")
    if enable_stoi:
        if _HAS_STOI and all_stoi_cat is not None:
            print(f"Average STOI: {all_stoi_cat.mean().item():.6f}")
        elif not _HAS_STOI:
            print("STOI skipped: install `pystoi` to enable.")
        else:
            print("STOI skipped: phase or reconstruction unavailable.")


def parse_args():
    p = argparse.ArgumentParser(description="Evaluate UNet denoiser on validation or test set")
    # Use config paths as defaults
    ckpt_default = os.path.join(CHECKPOINT_DIR, "unet_placeholder.pt")
    
    p.add_argument("--checkpoint", default=ckpt_default, help="Path to model checkpoint")
    p.add_argument("--data-root", default=PROCESSED_ROOT, help="Processed dataset root")
    p.add_argument("--split", default="val", choices=["val", "test"], help="Dataset split to evaluate (val or test)")
    p.add_argument("--reconstruct", action="store_true", help="Reconstruct waveforms using noisy phase and save to --out-dir")
    p.add_argument("--out-dir", default=f"{RESULTS_DIR}/recon", help="Output directory for reconstructed WAVs")
    p.add_argument("--no-stoi", action="store_true", help="Skip STOI computation even if pystoi is installed")
    p.add_argument("--dynamic", action="store_true", help="Use dynamic dataset generation (on-the-fly mixing)")
    
    # Use parse_known_args to handle Jupyter/Colab kernel arguments (like -f ...) gracefully
    args, _ = p.parse_known_args()
    return args

if __name__ == "__main__":
    args = parse_args()
    evaluate(
        args.checkpoint,
        args.data_root,
        split=args.split,
        reconstruct=args.reconstruct,
        out_dir=args.out_dir,
        enable_stoi=not args.no_stoi,
        dynamic=args.dynamic,
    )

In [ ]:
%%writefile utilis.py
import torch
import math

from config import get_config

cfg = get_config()


def _apply_light_postfilter(waveform: torch.Tensor, *,
                            sample_rate: int = None,
                            cutoff_hz: float = None,
                            strength: float = None) -> torch.Tensor:
    """Apply a gentle one-pole low-pass mix to reduce high-frequency hiss.

    waveform: (T,) or (B, T)
    strength: 0.0 -> unchanged, 1.0 -> fully low-passed
    """
    local_cfg = cfg
    sample_rate = sample_rate if sample_rate is not None else local_cfg.SAMPLE_RATE
    cutoff_hz = cutoff_hz if cutoff_hz is not None else local_cfg.POSTFILTER_CUTOFF_HZ
    strength = strength if strength is not None else local_cfg.POSTFILTER_STRENGTH

    if strength <= 0.0:
        return waveform

    cutoff_hz = max(1.0, min(float(cutoff_hz), (sample_rate * 0.5) - 1.0))
    strength = max(0.0, min(float(strength), 1.0))

    alpha = math.exp(-2.0 * math.pi * cutoff_hz / float(sample_rate))

    single = waveform.dim() == 1
    if single:
        waveform = waveform.unsqueeze(0)

    filtered = torch.empty_like(waveform)
    filtered[:, 0] = waveform[:, 0]
    for t in range(1, waveform.size(1)):
        filtered[:, t] = (1.0 - alpha) * waveform[:, t] + alpha * filtered[:, t - 1]

    out = (1.0 - strength) * waveform + strength * filtered
    return out.squeeze(0) if single else out


def _restore_nyquist_bin(mag: torch.Tensor, phase: torch.Tensor, *, n_fft: int, ref_mag: torch.Tensor = None):
    """If spectrogram has 512 bins from a 1024 FFT, append the missing Nyquist row.

    STFT with n_fft=1024 yields 513 bins. The training pipeline crops to 512
    for UNet convenience. Before ISTFT we re-attach the Nyquist bin.
    
    Improved: Instead of zero-padding (which causes artifacts), we copy
    the Nyquist bin from the reference noisy magnitude if available.
    """
    expected_bins = (n_fft // 2) + 1  # e.g., 1024 -> 513
    if mag.dim() == 4:
        # (B,1,F,T)
        freq_dim = 2
    else:
        # (B,F,T) or (F,T)
        freq_dim = 1 if mag.dim() == 3 else 0

    current_bins = mag.size(freq_dim)
    if current_bins == expected_bins:
        return mag, phase
    if current_bins != expected_bins - 1:
        raise ValueError(f"Unexpected freq bins: {current_bins}; expected {expected_bins} or {expected_bins-1}")

    pad_shape = list(mag.shape)
    pad_shape[freq_dim] = 1
    
    # Use reference magnitude for the Nyquist bin if available (preserve high-freq noise is better than hard zero)
    if ref_mag is not None:
        # Ensure ref_mag matches dimensions roughly or slice accordingly
        # ref_mag is likely (B, 1, 513, T) or similar.
        # We need the last bin from it.
        if ref_mag.dim() == mag.dim():
             # Access last bin along freq_dim
             nyquist_bin = torch.narrow(ref_mag, freq_dim, -1, 1)
             pad_mag = nyquist_bin
        else:
             pad_mag = torch.zeros(pad_shape, device=mag.device, dtype=mag.dtype)
    else:
        pad_mag = torch.zeros(pad_shape, device=mag.device, dtype=mag.dtype)

    mag = torch.cat([mag, pad_mag], dim=freq_dim)

    if phase is not None:
        pad_phase = torch.zeros(pad_shape, device=phase.device, dtype=phase.dtype)
        # If we copied magnitude, we should ideally copy phase too, but phase is passed separately usually.
        # For keeping it simple, 0 phase for Nyquist is acceptable provided magnitude isn't hard-cut zero.
        # (Ideally we'd accept ref_phase too, but ref_mag is the main perceptual fix).
        phase = torch.cat([phase, pad_phase], dim=freq_dim)

    return mag, phase


def reconstruct_waveform_from_mag_and_phase(mag: torch.Tensor, phase: torch.Tensor, *,
                                            n_fft: int = None,
                                            hop_length: int = None,
                                            win_length: int = None,
                                            window: torch.Tensor = None,
                                            use_log_mag: bool = None,
                                            ref_mag: torch.Tensor = None,
                                            ) -> torch.Tensor:
    """Reconstruct time waveform from magnitude and phase tensors.

    mag: (B, 1, F, T) or (F, T) or (B, F, T)
    phase: (B, F, T) or (F, T)
    ref_mag: Optional (B, 1, F_full, T) original noisy magnitude to borrow Nyquist bin from.

    Returns float32 waveform tensor (B, T_samples) if batch provided, else 1D tensor.
    """
    single = False
    if mag.dim() == 3 and mag.size(0) != 1 and mag.size(1) != 1:
        # mag shape (B, F, T)
        pass
    # normalize shapes to (B, F, T)
    if mag.dim() == 4:
        # (B, 1, F, T)
        mag = mag.squeeze(1)
    if mag.dim() == 2:
        mag = mag.unsqueeze(0)
        single = True

    if phase is None:
        raise ValueError("phase must be provided for noisy-phase reconstruction")
    if phase.dim() == 2:
        phase = phase.unsqueeze(0)

    # undo log1p if used during preprocessing/modeling
    local_cfg = cfg
    n_fft = n_fft if n_fft is not None else local_cfg.N_FFT
    hop_length = hop_length if hop_length is not None else local_cfg.HOP_LENGTH
    win_length = win_length if win_length is not None else local_cfg.WIN_LENGTH
    use_log_mag = use_log_mag if use_log_mag is not None else local_cfg.USE_LOG_MAG

    if use_log_mag:
        mag = torch.expm1(mag)
        
    # If ref_mag was provided in log1p domain (and use_log_mag is True), 
    # we must linearize it to match the 'mag' variable we just linearized above.
    # Note: ref_mag is optional, so check existence first.
    if ref_mag is not None and use_log_mag and use_log_mag is True:
        ref_mag_lin = torch.expm1(ref_mag)
    else:
        ref_mag_lin = ref_mag

    # re-attach missing Nyquist bin if model used 512 bins
    mag, phase = _restore_nyquist_bin(mag, phase, n_fft=n_fft, ref_mag=ref_mag_lin)

    # build complex STFT via polar form
    # mag and phase should be real tensors of same shape (B, F, T)
    complex_spec = torch.polar(mag, phase)

    # prepare window
    if window is None:
        window = torch.hann_window(win_length).to(complex_spec.device)

    # perform inverse STFT per sample
    waveforms = []
    for i in range(complex_spec.size(0)):
        spec = complex_spec[i].to(torch.complex64)
        waveform = torch.istft(spec,
                               n_fft=n_fft,
                               hop_length=hop_length,
                               win_length=win_length,
                               window=window,
                               length=None)
        waveforms.append(waveform)

    if single:
        return waveforms[0]
    return torch.stack(waveforms, dim=0)


def griffin_lim_reconstruct(mag: torch.Tensor, *,
                            n_iter: int = None,
                            n_fft: int = None,
                            hop_length: int = None,
                            win_length: int = None,
                            use_log_mag: bool = None,
                            ) -> torch.Tensor:
    """Reconstruct a waveform from a magnitude spectrogram using the Griffin-Lim algorithm.

    mag: (B, 1, F, T) or (B, F, T) or (F, T)
    Returns float32 waveform tensor (B, T_samples), or 1D tensor if single input.

    Parameters are read from config if not explicitly provided.
    """
    local_cfg = cfg
    n_iter      = n_iter      if n_iter      is not None else local_cfg.GRIFFIN_LIM_ITER
    n_fft       = n_fft       if n_fft       is not None else local_cfg.N_FFT
    hop_length  = hop_length  if hop_length  is not None else local_cfg.HOP_LENGTH
    win_length  = win_length  if win_length  is not None else local_cfg.WIN_LENGTH
    use_log_mag = use_log_mag if use_log_mag is not None else local_cfg.USE_LOG_MAG

    # Normalise shape to (B, F, T)
    single = False
    if mag.dim() == 4:
        mag = mag.squeeze(1)          # (B, 1, F, T) -> (B, F, T)
    if mag.dim() == 2:
        mag = mag.unsqueeze(0)        # (F, T) -> (1, F, T)
        single = True

    # Undo log1p if the spectrogram was stored in log-magnitude
    if use_log_mag:
        mag = torch.expm1(mag)

    # Re-attach Nyquist bin if model produced (N_FFT//2) bins instead of (N_FFT//2 + 1)
    dummy_phase = torch.zeros_like(mag)
    mag, _ = _restore_nyquist_bin(mag, dummy_phase, n_fft=n_fft)

    window = torch.hann_window(win_length).to(mag.device)

    waveforms = []
    for b in range(mag.size(0)):
        mag_b = mag[b]  # (F, T)

        # Initialise with random phase
        phase = torch.rand_like(mag_b) * 2 * math.pi - math.pi

        for _ in range(n_iter):
            # Build complex spectrogram from current magnitude + phase estimate
            complex_spec = torch.polar(mag_b, phase).to(torch.complex64)

            # ISTFT -> estimated waveform
            waveform = torch.istft(complex_spec,
                                   n_fft=n_fft,
                                   hop_length=hop_length,
                                   win_length=win_length,
                                   window=window)

            # STFT -> updated phase estimate
            new_complex = torch.stft(waveform,
                                     n_fft=n_fft,
                                     hop_length=hop_length,
                                     win_length=win_length,
                                     window=window,
                                     return_complex=True)
            phase = torch.angle(new_complex)

        waveforms.append(waveform)

    if single:
        return waveforms[0]
    return torch.stack(waveforms, dim=0)


def reconstruct_waveform_auto(mag: torch.Tensor, phase: torch.Tensor = None, *,
                              ref_mag: torch.Tensor = None,
                              prefer_input_phase: bool = None,
                              apply_postfilter: bool = None,
                              postfilter_cutoff_hz: float = None,
                              postfilter_strength: float = None,
                              apply_spectral_gate: bool = None,
                              spectral_gate_threshold: float = None,
                              apply_wiener: bool = None,
                              wiener_beta: float = None) -> torch.Tensor:
    """Auto-select reconstruction method and optionally apply anti-hiss postfilter.

    Policy:
      - If `phase` is available and `prefer_input_phase` is True: ISTFT with input phase.
      - Otherwise: Griffin-Lim reconstruction.

    Post-processing chain (all optional, executed in order):
      1. Spectral gating — zeroes out time-frequency bins below a power threshold
      2. Wiener-style post-filter — attenuates bins with low predicted SNR
      3. Light one-pole low-pass — reduces high-freq hiss in the time domain
    """
    local_cfg = cfg
    prefer_input_phase = prefer_input_phase if prefer_input_phase is not None else local_cfg.PREFER_INPUT_PHASE_RECON
    apply_postfilter = apply_postfilter if apply_postfilter is not None else local_cfg.APPLY_POSTFILTER
    apply_spectral_gate = apply_spectral_gate if apply_spectral_gate is not None else getattr(local_cfg, 'APPLY_SPECTRAL_GATE', True)
    spectral_gate_threshold = spectral_gate_threshold if spectral_gate_threshold is not None else getattr(local_cfg, 'SPECTRAL_GATE_THRESHOLD', 0.02)
    apply_wiener = apply_wiener if apply_wiener is not None else getattr(local_cfg, 'APPLY_WIENER_POSTFILTER', False)
    wiener_beta = wiener_beta if wiener_beta is not None else getattr(local_cfg, 'WIENER_BETA', 0.02)

    if phase is not None and prefer_input_phase:
        waveform = reconstruct_waveform_from_mag_and_phase(mag, phase, ref_mag=ref_mag)
    else:
        waveform = griffin_lim_reconstruct(mag)

    # --- Spectral post-processing ---
    if apply_spectral_gate or apply_wiener:
        waveform = _spectral_post_process(
            waveform,
            sample_rate=local_cfg.SAMPLE_RATE,
            apply_gate=apply_spectral_gate,
            gate_threshold=spectral_gate_threshold,
            apply_wiener=apply_wiener,
            wiener_beta=wiener_beta,
        )

    if apply_postfilter:
        waveform = _apply_light_postfilter(
            waveform,
            sample_rate=local_cfg.SAMPLE_RATE,
            cutoff_hz=postfilter_cutoff_hz,
            strength=postfilter_strength,
        )

    return waveform


# ---------------------------------------------------------------------------
# Spectral post-processing: gate + Wiener filter
# ---------------------------------------------------------------------------

def _spectral_post_process(waveform: torch.Tensor, *,
                           sample_rate: int = 16000,
                           n_fft: int = None,
                           hop_length: int = None,
                           win_length: int = None,
                           apply_gate: bool = True,
                           gate_threshold: float = 0.02,
                           apply_wiener: bool = False,
                           wiener_beta: float = 0.02) -> torch.Tensor:
    """Apply spectral-domain post-processing to remove residual static / rain noise.

    1. **Spectral gate**: Estimate a noise floor from the quietest 10% of frames,
       then zero out any T-F bin whose magnitude falls below ``gate_threshold``
       times that floor.  This removes the faint "rain" pattern without touching
       speech-dominant bins.
    2. **Wiener post-filter**: Compute a soft mask H = |S|^2 / (|S|^2 + beta)
       and multiply the complex spectrum by it.  Beta controls suppression
       strength (higher -> more aggressive).

    Both operate in the STFT domain and reconstruct via ISTFT, preserving length.
    """
    local_cfg = cfg
    n_fft = n_fft or local_cfg.N_FFT
    hop_length = hop_length or local_cfg.HOP_LENGTH
    win_length = win_length or local_cfg.WIN_LENGTH

    single = waveform.dim() == 1
    if single:
        waveform = waveform.unsqueeze(0)

    window = torch.hann_window(win_length, device=waveform.device)
    results = []

    for b in range(waveform.size(0)):
        sig = waveform[b]
        orig_len = sig.size(0)

        spec = torch.stft(sig, n_fft=n_fft, hop_length=hop_length,
                          win_length=win_length, window=window,
                          return_complex=True)
        mag = spec.abs()
        phase = spec.angle()

        if apply_gate:
            # Estimate noise floor from the quietest 10 % of frames
            frame_energy = mag.pow(2).mean(dim=0)  # (T,)
            k = max(1, int(frame_energy.size(0) * 0.10))
            noise_energy, _ = frame_energy.topk(k, largest=False)
            noise_floor = noise_energy.mean().sqrt()  # RMS of quiet frames

            # Soft gate: suppress bins below threshold * noise_floor
            threshold = gate_threshold * noise_floor
            gate_mask = (mag > threshold).float()
            # Smooth the mask slightly to avoid hard clickiness
            # Use a small 3x3 average pool if possible
            if gate_mask.dim() == 2:
                gm = gate_mask.unsqueeze(0).unsqueeze(0)
                gm = torch.nn.functional.avg_pool2d(gm, kernel_size=3, stride=1, padding=1)
                gate_mask = gm.squeeze(0).squeeze(0)
                gate_mask = (gate_mask > 0.5).float()
            mag = mag * gate_mask

        if apply_wiener:
            power = mag.pow(2)
            wiener_mask = power / (power + wiener_beta)
            mag = mag * wiener_mask

        spec_clean = torch.polar(mag, phase)
        recon = torch.istft(spec_clean, n_fft=n_fft, hop_length=hop_length,
                            win_length=win_length, window=window,
                            length=orig_len)
        results.append(recon)

    out = torch.stack(results, dim=0)
    return out.squeeze(0) if single else out



# Recommended Training Plan (Best Approach)

**Goal:** Maximum denoising quality with no residual static/rain/general noise.

| Stage | Mode | Loss | LR | Epochs | N2N | Purpose |
|-------|------|------|----|--------|-----|---------|
| 1 | Static | MSE | 2e-4 | 20 | True | Baseline convergence |
| 2 | Static | hybrid (α=0.3) | 1e-4 | 10 | True | Perceptual refinement |
| 3 | Dynamic | hybrid_l1 (α=0.4) | 8e-5 | 10 | True | Generalization + robust loss |
| 4 | Static | combined (λ1=100, λ2=1) | 5e-5 | 12 | **False** | Supervised cleanup (kills static/rain) |
| 5 | Dynamic | combined (λ1=50, λ2=1) | 2e-5 | 8 | **False** | Final polish |

**Total: ~60 epochs.** Best checkpoint selected by val loss at each stage.

### Key principles:
- **Stages 1-3 (N2N=True):** Learn robust denoising without needing clean targets
- **Stages 4-5 (N2N=False):** Supervised fine-tune removes residual noise that N2N can't eliminate
- **Loss progression:** MSE → hybrid → hybrid_l1 → combined (increasingly perceptual)
- **LR decay:** Start high, decay across stages for stable convergence
- After each stage, resume from `unet_best.pt` with `--reset-lr`

In [ ]:
# ============================================================
# STAGE 1: N2N Baseline (Static, MSE, 20 epochs)
# ============================================================
# Set NOISE2NOISE = True in config.py before running this stage
# This is the foundation — learns basic noise removal structure

!python train.py --loss mse --lr 2e-4 --epochs 20

In [ ]:
# ============================================================
# STAGE 2: Perceptual Refinement (Static, Hybrid, 10 epochs)
# ============================================================
# Resume from best Stage 1 checkpoint
# hybrid loss adds MR-STFT for better spectral quality

!python train.py \
    --resume checkpoints/unet_best.pt \
    --reset-lr --lr 1e-4 \
    --loss hybrid --alpha 0.3 \
    --epochs 30

In [ ]:
# ============================================================
# STAGE 3: Generalization (Dynamic, Hybrid L1, 10 epochs)
# ============================================================
# Dynamic mixing = new noise combos every epoch (better generalization)
# hybrid_l1 = L1 is more robust to noisy targets than MSE (better for N2N)
# Still N2N=True — noise pairs are now guaranteed independent (different files)

!python train.py \
    --resume checkpoints/unet_best.pt \
    --reset-lr --lr 8e-5 \
    --loss hybrid_l1 --alpha 0.4 \
    --dynamic \
    --epochs 40

In [ ]:
# ============================================================
# STAGE 4: Supervised Cleanup (Static, Combined, 12 epochs)
# ============================================================
# ⚠️ CRITICAL: Set NOISE2NOISE = False in config.py before this stage!
# This is where static/rain noise gets eliminated
# Combined loss = MSE + MR-STFT = aggressive denoising + perceptual quality

!python train.py \
    --resume checkpoints/unet_best.pt \
    --reset-lr --lr 5e-5 \
    --loss combined --lambda1 100 --lambda2 1 \
    --epochs 52

In [ ]:
# ============================================================
# STAGE 5: Final Polish (Dynamic, Combined, 8 epochs)
# ============================================================
# Still NOISE2NOISE = False
# Dynamic mixing for final generalization pass
# Lower LR for fine convergence without overshooting

!python train.py \
    --resume checkpoints/unet_best.pt \
    --reset-lr --lr 2e-5 \
    --loss combined --lambda1 50 --lambda2 1 \
    --dynamic \
    --epochs 60

In [ ]:
# ============================================================
# EVALUATE: Compare best checkpoint against baseline
# ============================================================

# 1. Baseline metrics (noisy input vs clean target)
!python baseline_metrics.py

# 2. Best model on validation set (with audio reconstruction)
!python evaluate.py \
    --checkpoint checkpoints/unet_best.pt \
    --split val \
    --reconstruct \
    --out-dir results/recon_best_val

# 3. Best model on test set
!python evaluate.py \
    --checkpoint checkpoints/unet_best.pt \
    --split test \
    --reconstruct \
    --out-dir results/recon_best_test

# 4. Visualize samples (spectrograms + audio)
!python visualize.py \
    --checkpoint checkpoints/unet_best.pt \
    --num-samples 5

## Stage 5 Results — Final Checkpoint (`unet_best_5.pt`)

| Metric | Value |
|--------|-------|
| Validation Samples | 1263 |
| Processed Samples | 1248 |
| Average PSNR | **32.664 dB** |
| Median PSNR | 32.689 dB |
| Average SNR | **11.030 dB** |
| Median SNR | 11.195 dB |
| Average MSE | 0.012573 |
| Average L1 | 0.044333 |
| Average LSD | 0.895355 |
| Average SSIM | **0.892163** |
| Average STOI | **0.909535** |

### Interpretation
- **PSNR 32.66 dB** — Good reconstruction fidelity; >30 dB is generally considered high quality for audio spectrogram recovery.
- **SNR 11.03 dB** — Decent noise suppression; meaningful improvement over a noisy baseline (typically 0–5 dB SNR).
- **SSIM 0.892** — High structural similarity; spectrogram structure is well preserved.
- **STOI 0.910** — Strong intelligibility score (max 1.0); speech intelligibility is well maintained post-denoising.
- **LSD 0.895** — Log-spectral distance; lower is better. Acceptable for a generalist denoiser.

### Next Steps
- Run `evaluate.py` on the **test split** to confirm these numbers generalise.
- Run `baseline_metrics.py` to compare against the raw noisy input baseline.
- Use `visualize.py` to inspect spectrograms for any remaining artefacts.
- If STOI or PSNR plateaus, consider a targeted fine-tune on the hardest noise types.